# 🚀 THỰC NGHIỆM ĐỐI CHỨNG: STAIR BASELINE (256D) vs STAIR DCD-GATED (256D) TRÊN CLOTHING
## 🏆 Amazon Clothing (39,387 Users, 23,033 Items, Độ thưa cực cao 99.97%) | Embedding Dim = 256D | Batch Size = 2048 | Patience = 30
---
### 🎯 NỘI DUNG VÀ MỤC TIÊU THỰC NGHIỆM:
1. **Đặc thù tập dữ liệu Clothing:**
   - Quy mô lớn với **39,387 người dùng** và **23,033 mặt hàng thời trang**.
   - Đồ thị tương tác có **độ thưa lên tới 99.97%** (cực kỳ thưa thớt), khiến việc lan truyền thông tin qua mạng GCN truyền thống dễ bị loãng tín hiệu.
2. **Hai thử nghiệm chính chạy thực tế ở không gian 256 chiều (256D):**
   - **PHẦN 1 (PART 1) — STAIR Baseline (256 chiều):** Huấn luyện Baseline thuần túy ở `256D` để kiểm tra năng lực mở rộng số chiều tự thân mà không có cải tiến đồ thị/tương phản.
   - **PHẦN 2 (PART 2) — STAIR DCD-Gated (256 chiều):** Triển khai cơ chế làm dày cạnh ảo đồng thuận kép (Hành vi giỏ hàng $\odot$ Tương đồng phong cách Text-Visual) kết hợp van an toàn Gating Residuals ở cùng không gian `256D`.
3. **PHẦN 3 (PART 3) — Bóc tách đóng góp khoa học (Ablation Study):**
   - Đối chiếu với mốc tham chiếu chuẩn Paper Baseline 64D (`NDCG@20 = 0.0398`).
   - Đo đạc chính xác $\Delta$ vs Base 64D và $\Delta$ vs Base 256D để xác định mức nhảy vọt thực sự của kiến trúc DCD-Gated ở cùng số chiều 256D.


In [ ]:
# Cell 1: Môi trường, Dependencies & Đồng bộ STAIR-Enhanced
import os, shutil, subprocess, sys

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
os.chdir('/kaggle/working')

# 1. Luôn clone hoặc đồng bộ cưỡng bức repository mới nhất từ origin/main
if os.path.exists(STAIR_DIR):
    print("Thư mục STAIR-Enhanced đã tồn tại. Đang đồng bộ cưỡng bức mã nguồn mới nhất...")
    try:
        subprocess.run(['git', '-C', STAIR_DIR, 'fetch', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', STAIR_DIR, 'reset', '--hard', 'origin/main'], check=True)
        print("✅ Đã reset về commit mới nhất của origin/main.")
    except Exception as e:
        print(f"Lỗi git fetch/reset ({e}), đang làm sạch và clone lại từ đầu...")
        shutil.rmtree(STAIR_DIR, ignore_errors=True)
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/HenryBui777/STAIR-Enhanced.git', STAIR_DIR
        ], check=True)
else:
    print("Cloning STAIR-Enhanced repository (branch main)...")
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/HenryBui777/STAIR-Enhanced.git', STAIR_DIR
    ], check=True)

for p in [STAIR_DIR, '/kaggle/working']:
    if p not in sys.path:
        sys.path.insert(0, p)

if os.path.exists(STAIR_DIR):
    os.chdir(STAIR_DIR)

# Ghi đảm bảo module breakthrough, v5_plus, baseline, runner và config tồn tại trên đĩa
os.makedirs(os.path.join(STAIR_DIR, 'models'), exist_ok=True)
os.makedirs(os.path.join(STAIR_DIR, 'configs'), exist_ok=True)
with open(os.path.join(STAIR_DIR, 'models', 'stair_breakthrough.py'), 'w', encoding='utf-8') as f:
    f.write('# -*- coding: utf-8 -*-\n"""\nmodels/stair_breakthrough.py\n============================\nBộ 3 Mô Hình Đột Phá Mở Rộng Từ STAIR-NE-NLGCL v5+ (v3-Refined):\n1. Method 1: DAN-TANS (Degree-Aware Noise & Topology-Aware Negative Scheduling)\n2. Method 2: DCD-Gated (Dual-Consensus Denoising & Gated Residuals)\n3. Method 3: APPNP-CrossModal (APPNP-Restart Propagation & Disentangled Cross-Modal Alignment)\n\nTối ưu chuyên biệt cho Amazon Baby & Amazon Sports (Zero OOM, VRAM < 1.2GB trên Kaggle T4).\n"""\n\nfrom typing import List, Optional, Tuple, Dict, Any\nimport math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\ntry:\n    from .stair_ne_nlgcl_v5_plus import STAIR_NE_NLGCL_v5_Plus\nexcept ImportError:\n    try:\n        from models.stair_ne_nlgcl_v5_plus import STAIR_NE_NLGCL_v5_Plus\n    except ImportError:\n        try:\n            from stair_ne_nlgcl_v5_plus import STAIR_NE_NLGCL_v5_Plus\n        except ImportError:\n            import importlib.util\n            import os\n            cand = os.path.join(os.path.dirname(__file__), "stair_ne_nlgcl_v5_plus.py")\n            if os.path.exists(cand):\n                spec = importlib.util.spec_from_file_location("stair_ne_nlgcl_v5_plus", cand)\n                mod = importlib.util.module_from_spec(spec)\n                spec.loader.exec_module(mod)\n                STAIR_NE_NLGCL_v5_Plus = mod.STAIR_NE_NLGCL_v5_Plus\n            else:\n                raise ImportError("Cannot find stair_ne_nlgcl_v5_plus.py to import STAIR_NE_NLGCL_v5_Plus!")\n\n__all__ = [\n    \'STAIR_DAN_TANS_Module\',\n    \'STAIR_DCD_Gated_Module\',\n    \'STAIR_APPNP_CrossModal_Module\',\n]\n\n\n# ═════════════════════════════════════════════════════════════════════════════\n# METHOD 1: DAN-TANS (Degree-Aware Noise & Topology-Aware Negative Scheduling)\n# ═════════════════════════════════════════════════════════════════════════════\nclass STAIR_DAN_TANS_Module(nn.Module):\n    """\n    Nâng cấp từ v5+:\n    - Đổi nhiễu tĩnh epsilon=0.08 thành nhiễu động theo bậc node d_i:\n      Head items (bậc cao) nhận nhiễu lớn hơn để chống over-smoothing;\n      Tail items (bậc thấp) nhận nhiễu nhỏ để bảo tồn biểu diễn mỏng manh.\n    - Đổi gamma_h=0.15 thành phạt thích ứng theo độ thưa:\n      Tăng mạnh phạt mẫu âm đối với các item đuôi dài để định hình biên quyết định rõ nét.\n    """\n    def __init__(\n        self,\n        n_users: Optional[int] = None,\n        n_items: Optional[int] = None,\n        tau: float = 0.20,\n        alpha_dir: float = 0.50,\n        eps_base: float = 0.08,\n        tau_thresh: float = 0.85,\n        lambda_cl: float = 0.010,\n        gamma_base: float = 0.15,\n        warmup_epochs: int = 50,\n    ):\n        super().__init__()\n        self.n_users = n_users\n        self.n_items = n_items\n        self.tau = tau\n        self.alpha_dir = alpha_dir\n        self.eps_base = eps_base\n        self.tau_thresh = tau_thresh\n        self.target_lambda = lambda_cl\n        self.gamma_base = gamma_base\n        self.warmup_epochs = warmup_epochs\n\n        self.current_epoch = 0\n        self.current_lambda = 0.0\n\n        # Node degree buffers (sẽ được gán khi khởi tạo)\n        self.register_buffer(\'user_degrees\', torch.zeros(n_users if n_users else 1))\n        self.register_buffer(\'item_degrees\', torch.zeros(n_items if n_items else 1))\n\n    def set_degrees(self, u_deg: torch.Tensor, i_deg: torch.Tensor):\n        self.user_degrees = u_deg.float().clamp(min=1.0)\n        self.item_degrees = i_deg.float().clamp(min=1.0)\n\n    def update_epoch(self, epoch: int):\n        self.current_epoch = epoch\n        if epoch <= self.warmup_epochs:\n            self.current_lambda = self.target_lambda * (float(epoch) / float(max(1, self.warmup_epochs)))\n        else:\n            self.current_lambda = self.target_lambda\n\n    def get_current_params(self) -> Tuple[float, float]:\n        return self.gamma_base, self.current_lambda\n\n    def get_current_hans_params(self) -> Tuple[float, float]:\n        return self.gamma_base, self.current_lambda\n\n    def get_adaptive_eps(self, degrees: torch.Tensor) -> torch.Tensor:\n        """\n        Nhiễu thích ứng bậc:\n        Node bậc cao nhận nhiễu lớn hơn (tới 1.4x), node bậc thấp nhận nhiễu dịu hơn (0.6x).\n        """\n        log_deg = torch.log(degrees + 1.0)\n        mean_deg = log_deg.mean()\n        std_deg = log_deg.std() + 1e-6\n        norm_deg = torch.tanh((log_deg - mean_deg) / std_deg) # [-1, 1]\n        eps_node = self.eps_base * (1.0 + 0.4 * norm_deg)\n        return eps_node.unsqueeze(-1) # (B, 1)\n\n    def get_adaptive_gamma(self, degrees: torch.Tensor) -> torch.Tensor:\n        """\n        Phạt mẫu âm thích ứng độ thưa (TANS):\n        Item càng ít tương tác (bậc nhỏ) -> gamma_h càng lớn (tới 2.0x) để buộc mô hình kéo xa ranh giới.\n        """\n        log_deg = torch.log(degrees + 1.0)\n        max_deg = log_deg.max() + 1e-6\n        gamma_node = self.gamma_base * (1.0 + (max_deg - log_deg) / max_deg)\n        return gamma_node # (B,)\n\n    def inject_adaptive_noise(self, h: torch.Tensor, beta: torch.Tensor, eps_adaptive: torch.Tensor) -> torch.Tensor:\n        if not self.training or self.eps_base <= 0.0:\n            return h\n        noise = torch.randn_like(h).abs()\n        noise = F.normalize(noise, p=2, dim=-1)\n        beta_w = beta.unsqueeze(0) if beta.dim() == 1 else beta\n        return h + eps_adaptive * (beta_w * torch.sign(h) * noise)\n\n    def forward(\n        self,\n        layer_embeds: List[torch.Tensor],\n        users: torch.Tensor,\n        positives: torch.Tensor,\n        beta: torch.Tensor,\n        item_modals: Optional[torch.Tensor] = None,\n    ) -> Tuple[torch.Tensor, float]:\n        users = users.view(-1)\n        positives = positives.view(-1)\n        device = layer_embeds[0].device\n        batch_size = users.size(0)\n\n        U_0, I_0 = torch.split(layer_embeds[0], [self.n_users, self.n_items])\n        U_1, I_1 = torch.split(layer_embeds[1], [self.n_users, self.n_items])\n\n        u_0 = U_0[users]\n        i_1 = I_1[positives]\n        i_0 = I_0[positives]\n        u_1 = U_1[users]\n\n        # Tính toán epsilon động theo bậc node trong batch\n        u_deg_b = self.user_degrees[users]\n        i_deg_b = self.item_degrees[positives]\n\n        eps_u = self.get_adaptive_eps(u_deg_b)\n        eps_i = self.get_adaptive_eps(i_deg_b)\n\n        u_0_t = F.normalize(self.inject_adaptive_noise(u_0, beta, eps_u), p=2, dim=-1)\n        i_1_t = F.normalize(self.inject_adaptive_noise(i_1, beta, eps_i), p=2, dim=-1)\n        i_0_t = F.normalize(self.inject_adaptive_noise(i_0, beta, eps_i), p=2, dim=-1)\n        u_1_t = F.normalize(self.inject_adaptive_noise(u_1, beta, eps_u), p=2, dim=-1)\n\n        # In-batch Dynamic Slicing + Hard MFNA\n        if item_modals is not None and self.tau_thresh < 1.0:\n            with torch.no_grad():\n                i_batch = item_modals[positives] if item_modals.size(0) != batch_size else item_modals\n                i_norm = F.normalize(i_batch, p=2, dim=-1)\n                sim_modal = torch.matmul(i_norm, i_norm.t())\n                mfna_mask = (sim_modal <= self.tau_thresh).float()\n        else:\n            mfna_mask = torch.ones((batch_size, batch_size), device=device)\n\n        diag_mask = ~torch.eye(batch_size, dtype=torch.bool, device=device)\n        valid_neg_mask = mfna_mask * diag_mask.float()\n\n        # Topology-aware gamma_h\n        gamma_i = self.get_adaptive_gamma(i_deg_b).unsqueeze(0) # (1, B)\n        gamma_u = self.get_adaptive_gamma(u_deg_b).unsqueeze(0) # (1, B)\n\n        # U -> I\n        pos_u2i = (u_0_t * i_1_t).sum(dim=-1) / self.tau\n        cos_u2i = torch.matmul(u_0_t, i_1_t.t())\n        sim_u2i = cos_u2i / self.tau\n        hans_u2i = 1.0 + gamma_i * torch.clamp(cos_u2i, min=0.0)\n        neg_u2i = valid_neg_mask * hans_u2i * torch.exp(sim_u2i)\n        loss_u2i = -(pos_u2i - torch.log(torch.exp(pos_u2i) + neg_u2i.sum(dim=-1) + 1e-8)).mean()\n\n        # I -> U\n        pos_i2u = (i_0_t * u_1_t).sum(dim=-1) / self.tau\n        cos_i2u = torch.matmul(i_0_t, u_1_t.t())\n        sim_i2u = cos_i2u / self.tau\n        hans_i2u = 1.0 + gamma_u * torch.clamp(cos_i2u, min=0.0)\n        neg_i2u = valid_neg_mask.t() * hans_i2u * torch.exp(sim_i2u)\n        loss_i2u = -(pos_i2u - torch.log(torch.exp(pos_i2u) + neg_i2u.sum(dim=-1) + 1e-8)).mean()\n\n        raw_loss = self.alpha_dir * loss_u2i + (1.0 - self.alpha_dir) * loss_i2u\n        total_loss = self.current_lambda * raw_loss\n        return total_loss, raw_loss.item()\n\n\n# ═════════════════════════════════════════════════════════════════════════════\n# METHOD 2: DCD-GATED (Dual-Consensus Denoising & Gated Residuals)\n# ═════════════════════════════════════════════════════════════════════════════\nclass STAIR_DCD_Gated_Module(STAIR_NE_NLGCL_v5_Plus):\n    """\n    Làm dày cạnh an toàn có cổng Gating chống suy thoái trên Baby:\n    - Ma trận cạnh ảo S_conf chỉ được kết nối khi có sự đồng thuận giữa:\n      Hành vi đồng mua (Ochiai Co-purchase) x Tương đồng đặc trưng ảnh/văn bản.\n    - Kênh cập nhật thặng dư đi qua cổng Gating phi tuyến g = sigmoid(W_g [H1 || S_conf H0]).\n      Nếu cạnh ảo bị nhiễu (như trên Baby), mạng tự động ép g -> 0 (an toàn 100%).\n    - Kế thừa toàn bộ InfoNCE Contrastive Loss của STAIR-NE-NLGCL v5+ (Linear HANS + Hard MFNA).\n    """\n    def __init__(\n        self,\n        n_users: Optional[int] = None,\n        n_items: Optional[int] = None,\n        embedding_dim: int = 256,\n        tau: float = 0.20,\n        alpha_dir: float = 0.50,\n        eps: float = 0.08,\n        tau_thresh: float = 0.85,\n        lambda_cl: float = 0.010,\n        gamma_h: float = 0.15,\n        warmup_epochs: int = 50,\n    ):\n        super().__init__(\n            n_users       = n_users,\n            n_items       = n_items,\n            tau           = tau,\n            alpha_dir     = alpha_dir,\n            eps           = eps,\n            tau_thresh    = tau_thresh,\n            lambda_cl     = lambda_cl,\n            gamma_h       = gamma_h,\n            warmup_epochs = warmup_epochs,\n        )\n        self.gate_fc = nn.Linear(embedding_dim * 2, embedding_dim)\n\n    def forward_gated_items(\n        self,\n        item_h0: torch.Tensor,\n        item_h1: torch.Tensor,\n        s_conf_sparse: Optional[torch.Tensor] = None,\n    ) -> torch.Tensor:\n        """Thực hiện làm dày cạnh an toàn có van kiểm soát."""\n        if s_conf_sparse is None:\n            return item_h1\n        virtual_h = torch.sparse.mm(s_conf_sparse, item_h0)\n        gate_input = torch.cat([item_h1, virtual_h], dim=-1)\n        gate = torch.sigmoid(self.gate_fc(gate_input))\n        return item_h1 + gate * virtual_h\n\n    def forward(\n        self,\n        layer_embeds: List[torch.Tensor],\n        users: torch.Tensor,\n        positives: torch.Tensor,\n        beta: torch.Tensor,\n        item_modals: Optional[torch.Tensor] = None,\n    ) -> Tuple[torch.Tensor, float]:\n        return super().forward(\n            layer_embeds=layer_embeds,\n            users=users,\n            positives=positives,\n            beta=beta,\n            item_modals=item_modals,\n        )\n\n\n# ═════════════════════════════════════════════════════════════════════════════\n# METHOD 3: APPNP-CROSSMODAL (APPNP-Restart Propagation & Disentangled CL)\n# ═════════════════════════════════════════════════════════════════════════════\nclass STAIR_APPNP_CrossModal_Module(STAIR_NE_NLGCL_v5_Plus):\n    """\n    1. APPNP-Restart Convolution:\n       H^(l) = (1 - alpha_restart) * (Adj @ H^(l-1) * beta) + alpha_restart * H^(0)\n       Bảo tồn 100% bản sắc đặc trưng gốc qua mọi tầng tích chập sâu.\n    2. Disentangled Cross-Modal Contrastive Alignment:\n       Kéo gần trực tiếp User Embedding H_u^(0) với Modal Feature M_i^(pos)\n       mà không nhét thêm bất kỳ cạnh bẩn nào vào đồ thị hành vi.\n    3. Kế thừa toàn bộ InfoNCE Contrastive Loss của STAIR-NE-NLGCL v5+.\n    """\n    def __init__(\n        self,\n        n_users: Optional[int] = None,\n        n_items: Optional[int] = None,\n        tau: float = 0.20,\n        alpha_dir: float = 0.50,\n        eps: float = 0.08,\n        tau_thresh: float = 0.85,\n        lambda_cl: float = 0.010,\n        lambda_cross: float = 0.005,\n        alpha_restart: float = 0.15,\n        gamma_h: float = 0.15,\n        warmup_epochs: int = 50,\n    ):\n        super().__init__(\n            n_users       = n_users,\n            n_items       = n_items,\n            tau           = tau,\n            alpha_dir     = alpha_dir,\n            eps           = eps,\n            tau_thresh    = tau_thresh,\n            lambda_cl     = lambda_cl,\n            gamma_h       = gamma_h,\n            warmup_epochs = warmup_epochs,\n        )\n        self.target_lambda_cross = lambda_cross\n        self.alpha_restart = alpha_restart\n        self.current_lambda_cross = 0.0\n\n    def update_epoch(self, epoch: int):\n        super().update_epoch(epoch)\n        ratio = float(min(epoch, self.warmup_epochs)) / float(max(1, self.warmup_epochs))\n        self.current_lambda_cross = self.target_lambda_cross * ratio\n\n    def forward_cross_modal(\n        self,\n        u_embeds: torch.Tensor,\n        item_modals: torch.Tensor,\n        users: torch.Tensor,\n        positives: torch.Tensor,\n    ) -> torch.Tensor:\n        """Tính hàm mất mát căn chỉnh trực tiếp sở thích người dùng với nội dung sản phẩm."""\n        u_b = F.normalize(u_embeds[users], p=2, dim=-1)\n        m_pos = F.normalize(item_modals[positives], p=2, dim=-1)\n\n        pos_sim = (u_b * m_pos).sum(dim=-1) / self.tau\n        all_sim = torch.matmul(u_b, m_pos.t()) / self.tau\n        loss_cross = -(pos_sim - torch.logsumexp(all_sim, dim=-1)).mean()\n        return self.current_lambda_cross * loss_cross\n\n    def forward(\n        self,\n        layer_embeds: List[torch.Tensor],\n        users: torch.Tensor,\n        positives: torch.Tensor,\n        beta: torch.Tensor,\n        item_modals: Optional[torch.Tensor] = None,\n    ) -> Tuple[torch.Tensor, float]:\n        return super().forward(\n            layer_embeds=layer_embeds,\n            users=users,\n            positives=positives,\n            beta=beta,\n            item_modals=item_modals,\n        )\n')
with open(os.path.join(STAIR_DIR, 'models', 'stair_ne_nlgcl_v5_plus.py'), 'w', encoding='utf-8') as f:
    f.write('# -*- coding: utf-8 -*-\n"""\nmodels/stair_ne_nlgcl_v5_plus.py\n=================================\nMô hình hoàn thiện tối ưu: STAIR-NE-NLGCL v5+ (v3-Refined)\nSelective Synergy & Minimalist Clean Architecture\n\nCore Design Pillars:\n────────────────────\n1. GNN Backbone trực tiếp (No Projection Head):\n   - Tương phản trực tiếp giữa H^(0) và H^(1) như SOTA v5.\n   - 100% thông lượng gradient InfoNCE truyền thẳng vào bảng embedding cơ sở E_u, E_i.\n2. True Sign-Preserving Spectral Perturbation (|noise| >= 0):\n   - h_tilde = h + eps * (beta * sign(h) * (|eta| / || |eta| ||_2))\n   - Bảo toàn tuyệt đối 100% góc phần tư không gian, triệt tiêu hiện tượng đảo pha tọa độ.\n3. Clean Linear HANS (Hardness-Aware Negative Scheduling):\n   - Phạt tuyến tính có ngưỡng: psi = 1.0 + gamma_h * clamp(cos_sim, min=0.0)\n   - gamma_h = 0.15 (thấp hơn v3), không làm co rút nhiệt độ hiệu dụng tau_eff = tau / (1 + gamma_h).\n4. Hard-Threshold MFNA (Modality False Negative Attenuation):\n   - Ngưỡng lọc cứng: Nếu S_modal > tau_thresh (0.85) -> mask = 0.0 (loại bỏ hoàn toàn mẫu âm giả).\n   - Ngược lại mask = 1.0. Tránh việc làm mờ gradient do soft clamping.\n5. In-batch Dynamic Slicing [B x B]:\n   - Chỉ tính toán lát cắt tương đồng modal trong batch, tiết kiệm 96.5% bộ nhớ, chống OOM.\n6. Constant Contrastive Weight with Linear Warmup:\n   - lambda_cl = 0.010 cố định (không Cosine decay làm suy kiệt lực đẩy ở cuối).\n   - Linear warmup 0 -> 0.010 trong 50 epochs đầu giúp BPR ổn định cấu trúc tô-pô.\n"""\n\nfrom typing import List, Optional, Tuple, Dict, Any\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n__all__ = [\'STAIR_NE_NLGCL_v5_Plus\']\n\n\nclass STAIR_NE_NLGCL_v5_Plus(nn.Module):\n    """\n    STAIR-NE-NLGCL v5+ Contrastive Learning Module.\n    Eliminates Projection Head, retains sign-preserving noise, applies linear HANS\n    and hard-threshold MFNA with constant contrastive pressure.\n    """\n\n    def __init__(\n        self,\n        n_users: Optional[int] = None,\n        n_items: Optional[int] = None,\n        tau: float = 0.20,\n        alpha_dir: float = 0.50,\n        eps: float = 0.08,           # Giảm nhẹ biên độ nhiễu xuống 0.08\n        tau_thresh: float = 0.85,    # Ngưỡng lọc cứng False Negatives\n        lambda_cl: float = 0.010,    # Cố định lực đẩy chống over-smoothing\n        gamma_h: float = 0.15,       # Phạt tuyến tính vừa phải (tránh co rút tau)\n        warmup_epochs: int = 50,     # Warmup tuyến tính cho lambda trong 50 epoch đầu\n    ):\n        super().__init__()\n        self.n_users = n_users\n        self.n_items = n_items\n        self.tau = tau\n        self.alpha_dir = alpha_dir\n        self.eps = eps\n        self.tau_thresh = tau_thresh\n        self.target_lambda = lambda_cl\n        self.gamma_h = gamma_h\n        self.warmup_epochs = warmup_epochs\n\n        self.current_epoch = 0\n        self.current_lambda = 0.0\n\n    def update_epoch(self, epoch: int):\n        """Warmup lambda từ 0 -> lambda_cl trong warmup_epochs đầu, sau đó cố định hoàn toàn."""\n        self.current_epoch = epoch\n        if epoch <= self.warmup_epochs:\n            self.current_lambda = self.target_lambda * (float(epoch) / float(max(1, self.warmup_epochs)))\n        else:\n            self.current_lambda = self.target_lambda\n\n    def get_current_params(self) -> Tuple[float, float]:\n        """Trả về tuple (gamma_h, current_lambda) phục vụ logging."""\n        return self.gamma_h, self.current_lambda\n\n    def get_current_hans_params(self) -> Tuple[float, float]:\n        """Alias tương thích ngược cho Coach v3."""\n        return self.gamma_h, self.current_lambda\n\n    def update_scheduler(self, current_cl_loss: float = 0.0, **kwargs):\n        """Alias tương thích ngược cho Coach gọi update_scheduler."""\n        pass\n\n    def inject_spectral_noise(self, h: torch.Tensor, beta: torch.Tensor) -> torch.Tensor:\n        """\n        Bơm nhiễu quang phổ bảo toàn hướng tuyệt đối với |noise|.\n        h_tilde = h + eps * (beta * sign(h) * (|noise| / || |noise| ||_2))\n        """\n        if not self.training or self.eps <= 0.0:\n            return h\n\n        noise = torch.randn_like(h).abs()\n        noise = F.normalize(noise, p=2, dim=-1)\n\n        beta_w = beta.unsqueeze(0) if beta.dim() == 1 else beta\n        return h + self.eps * (beta_w * torch.sign(h) * noise)\n\n    def forward(\n        self,\n        layer_embeds: List[torch.Tensor],\n        users: torch.Tensor,\n        positives: torch.Tensor,\n        beta: torch.Tensor,\n        item_modals: Optional[torch.Tensor] = None,\n    ) -> Tuple[torch.Tensor, float]:\n        """\n        Forward pass tính toán mất mát InfoNCE hai chiều với Linear HANS và Hard MFNA.\n\n        Args:\n            layer_embeds: [H^(0), H^(1), ...] từ FSC backbone, mỗi tensor có shape (N_u + N_i, D).\n            users: (B,) user indices trong mini-batch.\n            positives: (B,) positive item indices trong mini-batch.\n            beta: (D,) spectral propagation vector (1.0 - beta3).\n            item_modals: (B, D) hoặc (N_items, D) whitened modal features.\n\n        Returns:\n            Tuple: (total_loss có trọng số, raw_cl_loss dạng float)\n        """\n        users = users.view(-1)\n        positives = positives.view(-1)\n        device = layer_embeds[0].device\n        batch_size = users.size(0)\n\n        # ─────────────────────────────────────────────────────────────────\n        # 1. Trích xuất trực tiếp tầng 0 và tầng 1 (Không dùng Projection Head)\n        # ─────────────────────────────────────────────────────────────────\n        if self.n_users is not None and self.n_items is not None:\n            U_0, I_0 = torch.split(layer_embeds[0], [self.n_users, self.n_items])\n            U_1, I_1 = torch.split(layer_embeds[1], [self.n_users, self.n_items])\n            u_0 = U_0[users]\n            i_1 = I_1[positives]\n            i_0 = I_0[positives]\n            u_1 = U_1[users]\n        else:\n            num_u = layer_embeds[0].size(0) - (item_modals.size(0) if (item_modals is not None and item_modals.size(0) != batch_size) else 0)\n            u_0 = layer_embeds[0][users]\n            i_1 = layer_embeds[1][num_u + positives]\n            i_0 = layer_embeds[0][num_u + positives]\n            u_1 = layer_embeds[1][users]\n\n        # ─────────────────────────────────────────────────────────────────\n        # 2. Bơm nhiễu bảo toàn góc phần tư và chuẩn hóa L2\n        # ─────────────────────────────────────────────────────────────────\n        u_0_t = F.normalize(self.inject_spectral_noise(u_0, beta), p=2, dim=-1)\n        i_1_t = F.normalize(self.inject_spectral_noise(i_1, beta), p=2, dim=-1)\n        i_0_t = F.normalize(self.inject_spectral_noise(i_0, beta), p=2, dim=-1)\n        u_1_t = F.normalize(self.inject_spectral_noise(u_1, beta), p=2, dim=-1)\n\n        # ─────────────────────────────────────────────────────────────────\n        # 3. Dynamic Slicing + Ngưỡng cứng MFNA (Chống OOM & lọc triệt để False Negatives)\n        # ─────────────────────────────────────────────────────────────────\n        if item_modals is not None and self.tau_thresh < 1.0:\n            with torch.no_grad():\n                i_batch = item_modals[positives] if item_modals.size(0) != batch_size else item_modals\n                i_norm = F.normalize(i_batch, p=2, dim=-1)\n                sim_modal = torch.matmul(i_norm, i_norm.t())\n                # Ngưỡng cứng: nếu sim > tau_thresh loại bỏ hoàn toàn (mask = 0.0), ngược lại 1.0\n                mfna_mask = (sim_modal <= self.tau_thresh).float()\n        else:\n            mfna_mask = torch.ones((batch_size, batch_size), device=device)\n\n        diag_mask = ~torch.eye(batch_size, dtype=torch.bool, device=device)\n        valid_neg_mask = mfna_mask * diag_mask.float()\n\n        # ─────────────────────────────────────────────────────────────────\n        # 4. Hướng 1: User-to-Item (U_0 -> I_1) với phạt HANS tuyến tính\n        # ─────────────────────────────────────────────────────────────────\n        pos_u2i = (u_0_t * i_1_t).sum(dim=-1) / self.tau\n        cos_u2i = torch.matmul(u_0_t, i_1_t.t())\n        sim_u2i = cos_u2i / self.tau\n\n        # Phạt tuyến tính: psi = 1.0 + gamma_h * max(0, cos)\n        # Tuyệt đối không làm thay đổi nhiệt độ hiệu dụng tau_eff\n        hans_u2i = 1.0 + self.gamma_h * torch.clamp(cos_u2i, min=0.0)\n        neg_terms_u2i = valid_neg_mask * hans_u2i * torch.exp(sim_u2i)\n        loss_u2i = -(pos_u2i - torch.log(torch.exp(pos_u2i) + neg_terms_u2i.sum(dim=-1) + 1e-8)).mean()\n\n        # ─────────────────────────────────────────────────────────────────\n        # 5. Hướng 2: Item-to-User (I_0 -> U_1) với phạt HANS tuyến tính\n        # ─────────────────────────────────────────────────────────────────\n        pos_i2u = (i_0_t * u_1_t).sum(dim=-1) / self.tau\n        cos_i2u = torch.matmul(i_0_t, u_1_t.t())\n        sim_i2u = cos_i2u / self.tau\n\n        hans_i2u = 1.0 + self.gamma_h * torch.clamp(cos_i2u, min=0.0)\n        neg_terms_i2u = valid_neg_mask.t() * hans_i2u * torch.exp(sim_i2u)\n        loss_i2u = -(pos_i2u - torch.log(torch.exp(pos_i2u) + neg_terms_i2u.sum(dim=-1) + 1e-8)).mean()\n\n        # ─────────────────────────────────────────────────────────────────\n        # 6. Tổng hợp hàm mất mát có điều phối Warmup\n        # ─────────────────────────────────────────────────────────────────\n        raw_loss = self.alpha_dir * loss_u2i + (1.0 - self.alpha_dir) * loss_i2u\n        total_loss = self.current_lambda * raw_loss\n\n        return total_loss, raw_loss.item()\n')
with open(os.path.join(STAIR_DIR, 'main_stair_ne_nlgcl_v5_plus.py'), 'w', encoding='utf-8') as f:
    f.write('# -*- coding: utf-8 -*-\n"""\nmain_stair_ne_nlgcl_v5_plus.py — STAIR-NE-NLGCL v5+ (v3-Refined) Training Script\n================================================================================\nKế thừa trọn vẹn sự tinh gọn tối ưu của v5 (100% Direct Gradient Flow):\n1. Bỏ hoàn toàn Projection Head -> InfoNCE tác động trực tiếp vào H^(0) và H^(1).\n2. Bỏ hoàn toàn Regularized Diagonal Spectral Projector -> Loại bỏ ma sát tối ưu.\n3. Giữ nguyên Sign-Preserving Spectral Perturbation (|noise| >= 0) -> Bảo toàn góc phần tư 64D.\n4. Linear HANS (Hardness-Aware Negative Scheduling):\n   psi = 1.0 + gamma_h * clamp(cos_sim, min=0.0) với gamma_h = 0.15 (không làm méo tau_eff).\n5. Hard-Threshold MFNA (Modality False Negative Attenuation):\n   Nếu sim_modal > 0.85 -> mask = 0.0 (loại bỏ hoàn toàn near-duplicates), ngược lại 1.0.\n6. Constant Contrastive Pressure:\n   lambda_cl = 0.010 (warmup 0 -> 0.010 trong 50 epoch đầu, sau đó cố định 100%).\n\nUsage:\n    python main_stair_ne_nlgcl_v5_plus.py --config configs/Amazon2014Baby_550_MMRec.yaml\n    python main_stair_ne_nlgcl_v5_plus.py --config configs/Amazon2014Sports_550_MMRec.yaml\n"""\n\nimport math\nimport os\nimport sys\nimport types\nfrom typing import Dict, List, Optional, Tuple\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.utils.data\n\n# ── Compatibility Patch for torchdata in PyTorch 2.x / Python 3.12 / Kaggle ──\ntry:\n    import torchdata\n    import torchdata.datapipes as dp\nexcept Exception:\n    dp = None\n\nif dp is None or \'torchdata.datapipes\' not in sys.modules:\n    if \'torchdata\' not in sys.modules:\n        td = types.ModuleType(\'torchdata\')\n        sys.modules[\'torchdata\'] = td\n    else:\n        td = sys.modules[\'torchdata\']\n\n    dp = types.ModuleType(\'torchdata.datapipes\')\n    td.datapipes = dp\n    sys.modules[\'torchdata.datapipes\'] = dp\n\n# Ensure dp.iter and IterDataPipe exist\nif not hasattr(dp, \'iter\'):\n    iter_mod = types.ModuleType(\'torchdata.datapipes.iter\')\n    dp.iter = iter_mod\n    sys.modules[\'torchdata.datapipes.iter\'] = iter_mod\nif not hasattr(dp.iter, \'IterDataPipe\'):\n    class IterDataPipe(torch.utils.data.IterableDataset):\n        def __iter__(self):\n            return iter([])\n    dp.iter.IterDataPipe = IterDataPipe\n\n# Ensure dp.map and MapDataPipe exist\nif not hasattr(dp, \'map\'):\n    map_mod = types.ModuleType(\'torchdata.datapipes.map\')\n    dp.map = map_mod\n    sys.modules[\'torchdata.datapipes.map\'] = map_mod\nif not hasattr(dp.map, \'MapDataPipe\'):\n    class MapDataPipe(torch.utils.data.Dataset):\n        def __getitem__(self, idx):\n            raise NotImplementedError\n        def __len__(self):\n            return 0\n    dp.map.MapDataPipe = MapDataPipe\n\n# Ensure functional_datapipe decorator exists on dp\nif not hasattr(dp, \'functional_datapipe\'):\n    def functional_datapipe(name, enable_df_datapipes_support=False):\n        def decorator(cls):\n            def method(self, *args, **kwargs):\n                return cls(self, *args, **kwargs)\n            if hasattr(dp, \'iter\') and hasattr(dp.iter, \'IterDataPipe\'):\n                setattr(dp.iter.IterDataPipe, name, method)\n            if hasattr(dp, \'map\') and hasattr(dp.map, \'MapDataPipe\'):\n                setattr(dp.map.MapDataPipe, name, method)\n            try:\n                if hasattr(torch.utils.data, \'IterDataPipe\'):\n                    setattr(torch.utils.data.IterDataPipe, name, method)\n                if hasattr(torch.utils.data, \'MapDataPipe\'):\n                    setattr(torch.utils.data.MapDataPipe, name, method)\n            except Exception:\n                pass\n            return cls\n        return decorator\n    dp.functional_datapipe = functional_datapipe\n\nimport freerec\n\nfrom optimizers.Adam import AdamSEvo\nfrom optimizers.AdamW import AdamWSEvo\nfrom optimizers.utils import Smoother\n\nfrom models.stair_ne_nlgcl_v5_plus import STAIR_NE_NLGCL_v5_Plus\n\ntry:\n    from models.stair_breakthrough import (\n        STAIR_DAN_TANS_Module,\n        STAIR_DCD_Gated_Module,\n        STAIR_APPNP_CrossModal_Module,\n    )\nexcept ImportError:\n    try:\n        from stair_breakthrough import (\n            STAIR_DAN_TANS_Module,\n            STAIR_DCD_Gated_Module,\n            STAIR_APPNP_CrossModal_Module,\n        )\n    except ImportError:\n        STAIR_DAN_TANS_Module = None\n        STAIR_DCD_Gated_Module = None\n        STAIR_APPNP_CrossModal_Module = None\n\nfreerec.declare(version=\'0.8.5\')\n\n# ═════════════════════════════════════════════════════════════════════════════\n# Configuration Setup: STAIR Baseline + STAIR-NE-NLGCL v5+ (v3-Refined)\n# ═════════════════════════════════════════════════════════════════════════════\ncfg = freerec.parser.Parser()\n\n# ── STAIR Baseline Parameters ──\ncfg.add_argument("--embedding-dim", type=int, default=256,\n                 help="Latent vector embedding dimension D (default: 256)")\ncfg.add_argument("--num-layers", type=int, default=3,\n                 help="Number of layers for FSC/BSC (default: 3)")\ncfg.add_argument("--mfiles", type=str,\n                 default="textual_modality.pkl,visual_modality.pkl",\n                 help="Comma-separated modality feature files")\ncfg.add_argument("--num-neighbors", type=str, default=\'5-1\',\n                 help="kNN counts per modality, e.g. \'5-1\'")\ncfg.add_argument("--gamma", type=float, default=0.2,\n                 help="Spectral decay exponent for beta3 (default: 0.2)")\n\n# ── STAIR-NE-NLGCL v5+ & Breakthrough Parameters ──\ncfg.add_argument("--method", type=str, default="v5_plus",\n                 choices=["v5_plus", "dan_tans", "dcd_gated", "appnp_crossmodal"],\n                 help="Algorithm method: v5_plus, dan_tans, dcd_gated, appnp_crossmodal (default: v5_plus)")\ncfg.add_argument("--tau", type=float, default=0.20,\n                 help="Temperature tau for InfoNCE softmax (default: 0.20)")\ncfg.add_argument("--alpha-dir", type=float, default=0.50,\n                 help="Direction balance: alpha*L_{u->i} + (1-alpha)*L_{i->u} (default: 0.50)")\ncfg.add_argument("--eps", type=float, default=0.08,\n                 help="Sign-preserving noise amplitude epsilon (default: 0.08)")\ncfg.add_argument("--tau-thresh", type=float, default=0.85,\n                 help="Semantic similarity threshold for false negative masking (default: 0.85)")\ncfg.add_argument("--lambda-cl", type=float, default=0.010,\n                 help="Constant contrastive loss weight (default: 0.010)")\ncfg.add_argument("--gamma-h", type=float, default=0.15,\n                 help="Linear HANS hardness penalty coefficient (default: 0.15)")\ncfg.add_argument("--warmup-epochs", type=int, default=50,\n                 help="Warmup epochs for lambda (default: 50)")\n\n# ── LR Scheduler, Early Stopping & Checkpoint Selection ──\ncfg.add_argument("--lr-warmup-epochs", type=int, default=15,\n                 help="Warmup epochs for Learning Rate (default: 15)")\ncfg.add_argument("--min-lr", type=float, default=1e-6,\n                 help="Minimum LR after Cosine decay (default: 1e-6)")\ncfg.add_argument("--patience", type=int, default=30,\n                 help="Early stopping patience in epochs (default: 30)")\ncfg.add_argument("--target-metric", type=str, default="NDCG@20",\n                 help="Target metric for best checkpoint selection and early stopping (default: NDCG@20)")\n\n# ── Params riêng cho Hướng 3 (APPNP + CrossModal) ──\ncfg.add_argument("--alpha-restart", type=float, default=0.15, help="Hệ số teleport APPNP")\ncfg.add_argument("--lambda-cross", type=float, default=0.005, help="Trọng số cross-modal CL")\n\ncfg.set_defaults(\n    description="STAIR-NE-NLGCL-v5-Plus",\n    root="../../data",\n    dataset=\'Amazon2014Baby_550_MMRec\',\n    epochs=500,\n    batch_size=1024,\n    optimizer=\'adamwsevo\',\n    lr=1e-3,\n    weight_decay=0.1,\n    seed=1,\n    monitors=["Recall@10", "Recall@20", "NDCG@10", "NDCG@20"],\n    which4best="NDCG@20",\n)\ncfg.compile()\n\nif isinstance(cfg.mfiles, str):\n    cfg.mfiles = cfg.mfiles.split(\',\')\nif isinstance(cfg.num_neighbors, str):\n    cfg.num_neighbors = list(map(int, cfg.num_neighbors.split(\'-\')))\n\n# BSC Smoother spectral decay beta3\ncfg.beta3 = (\n    0.1 + 0.9 * (torch.arange(cfg.embedding_dim) / cfg.embedding_dim).pow(cfg.gamma)\n).to(cfg.device)\n\n\n# ═════════════════════════════════════════════════════════════════════════════\n# STAIR-NE-NLGCL v5+ (v3-Refined) & Breakthrough Architecture\n# ═════════════════════════════════════════════════════════════════════════════\nclass STAIR_NE_NLGCL_v5_Plus_Model(freerec.models.GenRecArch):\n    """\n    STAIR-NE-NLGCL v5+ (v3-Refined) & Breakthrough Architectures:\n    Combines STAIR Forward Stepwise Convolution with Clean Direct Contrastive Learning:\n    Supports 4 methods:\n    1. \'v5_plus\': Clean Baseline (100% Direct InfoNCE, Sign-Preserving Noise, Linear HANS, Hard MFNA)\n    2. \'dan_tans\': Degree-Aware Noise & Topology-Aware Negative Scheduling\n    3. \'dcd_gated\': Dual-Consensus Denoising with Dynamic Safety Gate\n    4. \'appnp_crossmodal\': APPNP Alpha-Restart Convolution & Disentangled Cross-Modal CL\n    """\n\n    def __init__(self, dataset: freerec.data.datasets.RecDataSet) -> None:\n        super().__init__(dataset)\n        self.num_layers = cfg.num_layers\n\n        self.User.add_module(\n            \'embeddings\', nn.Embedding(self.User.count, cfg.embedding_dim)\n        )\n        self.Item.add_module(\n            \'embeddings\', nn.Embedding(self.Item.count, cfg.embedding_dim)\n        )\n\n        self.register_buffer(\n            \'Adj\',\n            self.dataset.train().to_normalized_adj(normalization=\'sym\')\n        )\n        self.register_buffer(\'beta3\', cfg.beta3)\n\n        self.reset_parameters()\n        self.prepare(dataset.path)\n        self.criterion = freerec.criterions.BPRLoss(reduction=\'mean\')\n\n        # Khởi tạo Contrastive Module tương ứng với method\n        if cfg.method == \'dan_tans\' and STAIR_DAN_TANS_Module is not None:\n            self.ne_nlgcl_v5_plus = STAIR_DAN_TANS_Module(\n                n_users       = self.User.count,\n                n_items       = self.Item.count,\n                tau           = cfg.tau,\n                alpha_dir     = cfg.alpha_dir,\n                eps_base      = cfg.eps,\n                tau_thresh    = cfg.tau_thresh,\n                lambda_cl     = cfg.lambda_cl,\n                gamma_base    = cfg.gamma_h,\n                warmup_epochs = cfg.warmup_epochs,\n            )\n            edge_index_ui = self.dataset.train().to_bigraph(edge_type=\'u2i\')[\'u2i\'].edge_index\n            u_deg = edge_index_ui[0].bincount(minlength=self.User.count)\n            i_deg = edge_index_ui[1].bincount(minlength=self.Item.count)\n            self.ne_nlgcl_v5_plus.set_degrees(u_deg, i_deg)\n        elif cfg.method == \'dcd_gated\' and STAIR_DCD_Gated_Module is not None:\n            self.ne_nlgcl_v5_plus = STAIR_DCD_Gated_Module(\n                n_users       = self.User.count,\n                n_items       = self.Item.count,\n                embedding_dim = cfg.embedding_dim,\n                tau           = cfg.tau,\n                eps           = cfg.eps,\n                tau_thresh    = cfg.tau_thresh,\n                lambda_cl     = cfg.lambda_cl,\n                gamma_h       = cfg.gamma_h,\n                warmup_epochs = cfg.warmup_epochs,\n            )\n        elif cfg.method == \'appnp_crossmodal\' and STAIR_APPNP_CrossModal_Module is not None:\n            self.ne_nlgcl_v5_plus = STAIR_APPNP_CrossModal_Module(\n                n_users       = self.User.count,\n                n_items       = self.Item.count,\n                tau           = cfg.tau,\n                eps           = cfg.eps,\n                lambda_cl     = cfg.lambda_cl,\n                lambda_cross  = cfg.lambda_cross,\n                alpha_restart = cfg.alpha_restart,\n                warmup_epochs = cfg.warmup_epochs,\n            )\n        else:\n            self.ne_nlgcl_v5_plus = STAIR_NE_NLGCL_v5_Plus(\n                n_users       = self.User.count,\n                n_items       = self.Item.count,\n                tau           = cfg.tau,\n                alpha_dir     = cfg.alpha_dir,\n                eps           = cfg.eps,\n                tau_thresh    = cfg.tau_thresh,\n                lambda_cl     = cfg.lambda_cl,\n                gamma_h       = cfg.gamma_h,\n                warmup_epochs = cfg.warmup_epochs,\n            )\n\n        self.last_cl_loss: Optional[float] = None\n        self.s_conf_sparse: Optional[torch.Tensor] = None\n\n    def reset_parameters(self):\n        for m in self.modules():\n            if isinstance(m, nn.Linear):\n                nn.init.kaiming_normal_(m.weight)\n                if m.bias is not None:\n                    nn.init.constant_(m.bias, 0.)\n            elif isinstance(m, nn.Embedding):\n                nn.init.normal_(m.weight, std=1.e-4)\n            elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):\n                nn.init.constant_(m.weight, 1.)\n                nn.init.constant_(m.bias, 0.)\n\n    def marked_params(self):\n        params = [\n            {\n                \'params\': self.User.parameters(),\n                \'smoother\': None\n            },\n            {\n                \'params\': self.Item.parameters(), \n                \'smoother\': Smoother(self.mAdj, beta=cfg.beta3, L=cfg.num_layers, aggr=\'neumann\')\n            },\n        ]\n        return params\n\n    def whitening(self, feats: torch.Tensor):\n        if not isinstance(feats, torch.Tensor):\n            feats = torch.tensor(feats, dtype=torch.float32)\n        else:\n            feats = feats.float()\n        feats = feats - feats.mean(0, keepdim=True)\n        feats, _, _ = torch.linalg.svd(feats, full_matrices=False)\n        if feats.size(1) < cfg.embedding_dim:\n            reps = math.ceil(cfg.embedding_dim / feats.size(1))\n            scale = math.sqrt(feats.size(1) / cfg.embedding_dim)\n            feats = (feats.repeat(1, reps)[:, :cfg.embedding_dim]) * scale\n        else:\n            feats = feats[:, :cfg.embedding_dim]\n        return feats * math.sqrt(self.Item.count / cfg.embedding_dim)\n\n    def get_knn_graph(self, features: torch.Tensor, k: int = 5):\n        if not isinstance(features, torch.Tensor):\n            features = torch.tensor(features, dtype=torch.float32)\n        else:\n            features = features.float()\n        features = F.normalize(features, dim=-1)\n        sim = features @ features.t()\n        sim.fill_diagonal_(-10.)\n        edge_index, _ = freerec.graph.get_knn_graph(\n            sim, k, symmetric=False\n        )\n        return edge_index\n\n    def prepare(self, path: str):\n        from freerec.utils import import_pickle\n\n        mfeats = []\n        for mfile in cfg.mfiles:\n            mpath = os.path.join(path, mfile)\n            if not os.path.exists(mpath):\n                for cand in [\n                    os.path.join(cfg.root, cfg.dataset, mfile),\n                    os.path.join("/kaggle/data", cfg.dataset, mfile),\n                    os.path.join("/kaggle/data/Processed", cfg.dataset, mfile),\n                    os.path.join("/kaggle/working/STAIR/data", cfg.dataset, mfile),\n                    os.path.join("/kaggle/working/STAIR-Enhanced/data", cfg.dataset, mfile),\n                    os.path.join("data", cfg.dataset, mfile),\n                    os.path.join("data/Processed", cfg.dataset, mfile),\n                ]:\n                    if os.path.exists(cand):\n                        mpath = cand\n                        break\n            mfeats.append(import_pickle(mpath))\n\n        edge_index = torch.cat(\n            [self.get_knn_graph(feats, k)\n             for feats, k in zip(mfeats, cfg.num_neighbors)],\n            dim=1\n        )\n        edge_weight = torch.ones_like(edge_index[0], dtype=torch.float)\n        edge_index, edge_weight = freerec.graph.coalesce(\n            edge_index, edge_weight, reduce=\'sum\'\n        )\n        edge_index, edge_weight = freerec.graph.to_undirected(\n            edge_index, edge_weight, reduce=\'max\'\n        )\n        edge_index, edge_weight = freerec.graph.to_normalized(\n            edge_index, edge_weight, normalization=\'sym\'\n        )\n        mAdj = torch.sparse_coo_tensor(\n            edge_index, edge_weight,\n            size=(self.Item.count, self.Item.count)\n        )\n        self.register_buffer(\'mAdj\', mAdj.to_sparse_csr())\n\n        # Whitened modal feature initialization\n        mfeats_w = [\n            self.whitening(mfeat) * k\n            for mfeat, k in zip(mfeats, cfg.num_neighbors)\n        ]\n        mfeats_init = sum(mfeats_w).div(sum(cfg.num_neighbors))\n        self.Item.embeddings.weight.data.copy_(mfeats_init)\n\n        edge_index_ui = self.dataset.train().to_bigraph(\n            edge_type=\'u2i\'\n        )[\'u2i\'].edge_index\n        edge_index_ui, edge_weight_ui = freerec.graph.to_normalized(\n            edge_index_ui, normalization=\'left\'\n        )\n        R = torch.sparse_coo_tensor(\n            edge_index_ui, edge_weight_ui,\n            size=(self.User.count, self.Item.count)\n        ).to_sparse_csr()\n        user_profiles_init = R @ mfeats_init\n        self.User.embeddings.weight.data.copy_(user_profiles_init)\n\n        # Register raw modal features for In-batch Dynamic False Negative Attenuation\n        self.register_buffer(\'item_modals_raw\', mfeats_init.detach().clone())\n\n        # ── Setup Bổ sung cho từng Method ──\n        raw_edge_ui = self.dataset.train().to_bigraph(edge_type=\'u2i\')[\'u2i\'].edge_index\n        u_deg_cpu = raw_edge_ui[0].bincount(minlength=self.User.count)\n        i_deg_cpu = raw_edge_ui[1].bincount(minlength=self.Item.count)\n        u_deg = u_deg_cpu.to(cfg.device)\n        i_deg = i_deg_cpu.to(cfg.device)\n\n        if cfg.method == \'dan_tans\' and hasattr(self.ne_nlgcl_v5_plus, \'set_degrees\'):\n            self.ne_nlgcl_v5_plus.set_degrees(u_deg, i_deg)\n        elif cfg.method == \'dcd_gated\':\n            with torch.no_grad():\n                # ── True Zero-OOM Chunked Dual Consensus Engine (Thích ứng từ Baby -> Electronics 63K) ──\n                # Xử lý theo từng khối chunk_size=2000 items: RAM đỉnh < 5GB, triệt tiêu 100% nguy cơ OOM / SIGKILL!\n                i_norm = F.normalize(mfeats_init.cpu(), p=2, dim=-1)\n                edge_weight_ui_ones = torch.ones_like(raw_edge_ui[0], dtype=torch.float)\n                R_csr = torch.sparse_coo_tensor(\n                    raw_edge_ui.cpu(), edge_weight_ui_ones.cpu(),\n                    size=(self.User.count, self.Item.count)\n                ).to_sparse_csr()\n\n                chunk_size = 2000\n                collected_rows = []\n                collected_cols = []\n                collected_vals = []\n\n                raw_items = raw_edge_ui[1].cpu()\n                raw_users = raw_edge_ui[0].cpu()\n                item_sort_perm = torch.argsort(raw_items)\n                sorted_items = raw_items[item_sort_perm]\n                sorted_users = raw_users[item_sort_perm]\n                item_ptrs = torch.searchsorted(sorted_items, torch.arange(self.Item.count + 1))\n\n                for start_idx in range(0, self.Item.count, chunk_size):\n                    end_idx = min(start_idx + chunk_size, self.Item.count)\n                    cur_chunk_len = end_idx - start_idx\n                    p_start = item_ptrs[start_idx].item()\n                    p_end = item_ptrs[end_idx].item()\n\n                    if p_end > p_start:\n                        chunk_item_indices = sorted_items[p_start:p_end] - start_idx\n                        chunk_user_indices = sorted_users[p_start:p_end]\n                        Rt_chunk = torch.sparse_coo_tensor(\n                            torch.stack([chunk_item_indices, chunk_user_indices]),\n                            torch.ones(p_end - p_start, dtype=torch.float),\n                            size=(cur_chunk_len, self.User.count)\n                        ).to_sparse_csr()\n                        co_chunk = torch.sparse.mm(Rt_chunk, R_csr).to_dense()\n                    else:\n                        co_chunk = torch.zeros((cur_chunk_len, self.Item.count), dtype=torch.float)\n\n                    deg_row = i_deg_cpu[start_idx:end_idx].float().unsqueeze(1)\n                    deg_col = i_deg_cpu.float().unsqueeze(0)\n                    deg_norm_chunk = torch.sqrt(deg_row * deg_col).clamp(min=1.0)\n                    ochiai_chunk = co_chunk / deg_norm_chunk\n\n                    sim_m_chunk = torch.clamp(\n                        torch.matmul(i_norm[start_idx:end_idx], i_norm.t()),\n                        min=0.0\n                    )\n\n                    consensus_chunk = ochiai_chunk * sim_m_chunk\n                    diag_rows = torch.arange(cur_chunk_len)\n                    diag_cols = torch.arange(start_idx, end_idx)\n                    consensus_chunk[diag_rows, diag_cols] = 0.0\n\n                    topk_val, topk_idx = torch.topk(consensus_chunk, k=min(3, self.Item.count), dim=-1)\n                    valid_mask = topk_val > 0.05\n\n                    if valid_mask.any():\n                        row_ids = torch.arange(start_idx, end_idx).unsqueeze(1).expand(-1, topk_val.size(1))[valid_mask]\n                        col_ids = topk_idx[valid_mask]\n                        val_ids = topk_val[valid_mask]\n                        collected_rows.append(row_ids)\n                        collected_cols.append(col_ids)\n                        collected_vals.append(val_ids)\n\n                if len(collected_rows) > 0:\n                    all_rows = torch.cat(collected_rows, dim=0)\n                    all_cols = torch.cat(collected_cols, dim=0)\n                    all_vals = torch.cat(collected_vals, dim=0)\n\n                    conf_idx = torch.stack([all_rows, all_cols], dim=0)\n                    conf_idx, edge_val = freerec.graph.to_normalized(conf_idx, all_vals, normalization=\'sym\')\n                    self.s_conf_sparse = torch.sparse_coo_tensor(\n                        conf_idx.to(cfg.device), edge_val.to(cfg.device),\n                        size=(self.Item.count, self.Item.count),\n                        device=cfg.device\n                    ).coalesce()\n                    print(f"[{cfg.dataset}] ✅ DCD-Gated: Khởi tạo thành công {len(edge_val)} cạnh ảo đồng thuận cao (Chunked Zero-OOM Engine, Device={cfg.device}).")\n                else:\n                    self.s_conf_sparse = None\n                    print(f"[{cfg.dataset}] ℹ️ DCD-Gated: Không tìm thấy cạnh ảo nào vượt ngưỡng đồng thuận > 0.05.")\n\n    def sure_trainpipe(self, batch_size: int):\n        return (\n            self.dataset.train()\n            .shuffled_pairs_source()\n            .gen_train_sampling_neg_(num_negatives=1)\n            .batch_(batch_size)\n            .tensor_()\n        )\n\n    def encode(self) -> Tuple[torch.Tensor, torch.Tensor, List[torch.Tensor]]:\n        """\n        Forward Stepwise Convolution with Layer Intermediates capture:\n        Returns:\n            userEmbds:    (N_u, D) final aggregated user representations\n            itemEmbds:    (N_i, D) final aggregated item representations\n            layer_embeds: [H^0, H^1, ..., H^L] per-layer representations\n        """\n        allEmbds = torch.cat(\n            (self.User.embeddings.weight, self.Item.embeddings.weight),\n            dim=0,\n        )\n\n        layer_embeds = [allEmbds]\n        features = allEmbds\n        smoothed = allEmbds\n\n        beta = (1.0 - self.beta3).to(allEmbds.device)\n        norm_correction = 1.0 - beta ** (self.num_layers + 1)\n        alpha_restart = getattr(self.ne_nlgcl_v5_plus, \'alpha_restart\', 0.0) if cfg.method == \'appnp_crossmodal\' else 0.0\n\n        for _ in range(self.num_layers):\n            if alpha_restart > 0.0:\n                features = (1.0 - alpha_restart) * (self.Adj @ features * beta) + alpha_restart * allEmbds\n            else:\n                features = self.Adj @ features * beta\n            smoothed = smoothed + features\n            layer_embeds.append(features)\n\n        avgEmbds = smoothed.mul(1.0 - beta).div(norm_correction)\n        userEmbds, itemEmbds = torch.split(\n            avgEmbds, (self.User.count, self.Item.count)\n        )\n\n        # Gated refinement cho DCD-Gated\n        if cfg.method == \'dcd_gated\' and hasattr(self, \'s_conf_sparse\') and self.s_conf_sparse is not None:\n            if hasattr(self.ne_nlgcl_v5_plus, \'forward_gated_items\'):\n                U_0, I_0 = torch.split(layer_embeds[0], [self.User.count, self.Item.count])\n                U_1, I_1 = torch.split(layer_embeds[1], [self.User.count, self.Item.count])\n                I_1_refined = self.ne_nlgcl_v5_plus.forward_gated_items(I_0, I_1, self.s_conf_sparse)\n                layer_embeds[1] = torch.cat([U_1, I_1_refined], dim=0)\n\n        return userEmbds, itemEmbds, layer_embeds\n\n    def encode_for_eval(self) -> Tuple[torch.Tensor, torch.Tensor]:\n        """Evaluation encode function (zero overhead)."""\n        allEmbds = torch.cat(\n            (self.User.embeddings.weight, self.Item.embeddings.weight),\n            dim=0,\n        )\n        features = allEmbds\n        smoothed = allEmbds\n        beta = (1.0 - self.beta3).to(allEmbds.device)\n        norm_correction = 1.0 - beta ** (self.num_layers + 1)\n        alpha_restart = getattr(self.ne_nlgcl_v5_plus, \'alpha_restart\', 0.0) if cfg.method == \'appnp_crossmodal\' else 0.0\n\n        for _ in range(self.num_layers):\n            if alpha_restart > 0.0:\n                features = (1.0 - alpha_restart) * (self.Adj @ features * beta) + alpha_restart * allEmbds\n            else:\n                features = self.Adj @ features * beta\n            smoothed = smoothed + features\n\n        avgEmbds = smoothed.mul(1.0 - beta).div(norm_correction)\n        return torch.split(avgEmbds, (self.User.count, self.Item.count))\n\n    def fit(self, data: Dict[freerec.data.fields.Field, torch.Tensor]):\n        """\n        Training step:\n        L_total = L_BPR + lambda_cl(t) * L_NE-NLGCL_v5+ (+ L_cross nếu APPNP)\n        """\n        userEmbds, itemEmbds, layer_embeds = self.encode()\n\n        users     = data[self.User]\n        positives = data[self.Item]\n        negatives = data[self.INeg]\n\n        # 1. Pairwise BPR Ranking Loss\n        rec_loss = self.criterion(\n            torch.einsum(\'BKD,BKD->BK\', userEmbds[users], itemEmbds[positives]),\n            torch.einsum(\'BKD,BKD->BK\', userEmbds[users], itemEmbds[negatives]),\n        )\n\n        # 2. STAIR-NE-NLGCL v5+ / Breakthrough Contrastive Loss\n        if self.training:\n            beta = (1.0 - self.beta3).to(userEmbds.device)\n            i_mod = self.item_modals_raw if hasattr(self, \'item_modals_raw\') else None\n\n            if cfg.method == \'appnp_crossmodal\' and hasattr(self.ne_nlgcl_v5_plus, \'forward_cross_modal\') and i_mod is not None:\n                cross_loss = self.ne_nlgcl_v5_plus.forward_cross_modal(userEmbds, i_mod, users, positives)\n                weighted_cl_loss, raw_cl_loss = self.ne_nlgcl_v5_plus(\n                    layer_embeds = layer_embeds,\n                    users        = users,\n                    positives    = positives,\n                    beta         = beta,\n                    item_modals  = i_mod,\n                )\n                self.last_cl_loss = raw_cl_loss\n                return rec_loss + weighted_cl_loss + cross_loss\n\n            weighted_cl_loss, raw_cl_loss = self.ne_nlgcl_v5_plus(\n                layer_embeds = layer_embeds,\n                users        = users,\n                positives    = positives,\n                beta         = beta,\n                item_modals  = i_mod,\n            )\n            self.last_cl_loss = raw_cl_loss\n            return rec_loss + weighted_cl_loss\n\n        return rec_loss\n\n    def reset_ranking_buffers(self):\n        userEmbds, itemEmbds = self.encode_for_eval()\n        self.ranking_buffer = {\n            self.User: userEmbds.detach().clone(),\n            self.Item: itemEmbds.detach().clone(),\n        }\n\n    def recommend_from_full(self, data):\n        userEmbds = self.ranking_buffer[self.User][data[self.User]]\n        itemEmbds = self.ranking_buffer[self.Item]\n        return torch.einsum(\'BKD,ND->BN\', userEmbds, itemEmbds)\n\n    def recommend_from_pool(self, data):\n        userEmbds = self.ranking_buffer[self.User][data[self.User]]\n        itemEmbds = self.ranking_buffer[self.Item][data[self.IUnseen]]\n        return torch.einsum(\'BKD,BKD->BK\', userEmbds, itemEmbds)\n\n\n# ═════════════════════════════════════════════════════════════════════════════\n# Coach Class for STAIR-NE-NLGCL v5+ & Breakthrough Methods\n# ═════════════════════════════════════════════════════════════════════════════\nclass CoachForSTAIR_NE_NLGCL_v5_Plus(freerec.launcher.Coach):\n\n    def __init__(self, *args, **kwargs):\n        super().__init__(*args, **kwargs)\n        self.best_ndcg20 = -1.0\n        self.best_epoch = 0\n        self.patience_counter = 0\n        self.patience = getattr(self.cfg, \'patience\', 30)\n        self.lr_warmup_epochs = getattr(self.cfg, \'lr_warmup_epochs\', 15)\n        self.min_lr = getattr(self.cfg, \'min_lr\', 1e-6)\n\n    def adjust_learning_rate(self, epoch: int) -> float:\n        base_lr = self.cfg.lr\n        min_lr = self.min_lr\n        warmup = self.lr_warmup_epochs\n        total = self.cfg.epochs\n\n        if epoch < warmup:\n            current_lr = min_lr + (base_lr - min_lr) * float(epoch + 1) / float(max(1, warmup))\n        else:\n            progress = float(epoch + 1 - warmup) / float(max(1, total - warmup))\n            current_lr = min_lr + 0.5 * (base_lr - min_lr) * (1.0 + math.cos(math.pi * progress))\n\n        for param_group in self.optimizer.param_groups:\n            param_group[\'lr\'] = current_lr\n        return current_lr\n\n    def set_optimizer(self):\n        if self.cfg.optimizer.lower() == \'adamwsevo\':\n            self.optimizer = AdamWSEvo(\n                self.model.marked_params(), lr=self.cfg.lr,\n                betas=(self.cfg.beta1, self.cfg.beta2),\n                weight_decay=self.cfg.weight_decay,\n            )\n        elif self.cfg.optimizer.lower() == \'adamsevo\':\n            self.optimizer = AdamSEvo(\n                self.model.marked_params(), lr=self.cfg.lr,\n                betas=(self.cfg.beta1, self.cfg.beta2),\n                weight_decay=self.cfg.weight_decay,\n            )\n        elif self.cfg.optimizer.lower() == \'adamw\':\n            self.optimizer = torch.optim.AdamW(\n                self.model.marked_params(), lr=self.cfg.lr,\n                betas=(self.cfg.beta1, self.cfg.beta2),\n                weight_decay=self.cfg.weight_decay,\n            )\n        elif self.cfg.optimizer.lower() == \'adam\':\n            self.optimizer = torch.optim.Adam(\n                self.model.marked_params(), lr=self.cfg.lr,\n                betas=(self.cfg.beta1, self.cfg.beta2),\n                weight_decay=self.cfg.weight_decay,\n            )\n        else:\n            raise NotImplementedError(\n                f"CoachForSTAIR_NE_NLGCL_v5_Plus does not support {self.cfg.optimizer} optimizer"\n            )\n\n    def train_per_epoch(self, epoch: int):\n        self.model.train()\n        current_lr = self.adjust_learning_rate(epoch)\n        self.model.ne_nlgcl_v5_plus.update_epoch(epoch + 1)\n        total_cl_loss = 0.0\n        cl_batches = 0\n\n        for data in self.dataloader:\n            data = self.dict_to_device(data)\n            loss = self.model(data)\n\n            self.optimizer.zero_grad()\n            loss.backward()\n            self.optimizer.step()\n\n            self.monitor(\n                loss.item(), n=len(data[self.User]),\n                reduction="mean", mode=\'train\', pool=[\'LOSS\'],\n            )\n\n            if hasattr(self.model, \'last_cl_loss\') and self.model.last_cl_loss is not None:\n                total_cl_loss += self.model.last_cl_loss\n                cl_batches += 1\n\n        if cl_batches > 0:\n            avg_cl_loss = total_cl_loss / float(cl_batches)\n            gamma_h = getattr(self.model.ne_nlgcl_v5_plus, \'gamma_h\', getattr(self.model.ne_nlgcl_v5_plus, \'gamma_base\', 0.15))\n            curr_lambda = getattr(self.model.ne_nlgcl_v5_plus, \'current_lambda\', 0.0)\n            if (epoch + 1) % 10 == 0 or epoch == 0 or (epoch + 1) == self.cfg.epochs:\n                method_name = getattr(self.cfg, \'method\', \'v5_plus\')\n                print(\n                    f"  [{method_name.upper()} Epoch {epoch + 1:03d}] LR: {current_lr:.6e} | "\n                    f"gamma_h: {gamma_h:.4f} | lambda: {curr_lambda:.5f} | avg_cl_loss: {avg_cl_loss:.6f}"\n                )\n\n    def evaluate(self, epoch: int = 0, mode: str = \'valid\'):\n        super().evaluate(epoch, mode=mode)\n        if mode == \'valid\':\n            try:\n                if not hasattr(self, \'_patience_counter\'):\n                    self._patience_counter = 0\n                    self._best_ep = getattr(self, \'best_epoch\', 0)\n\n                cur_best_ep = getattr(self, \'best_epoch\', 0)\n                if cur_best_ep == epoch:\n                    self._patience_counter = 0\n                    self._best_ep = epoch\n                else:\n                    self._patience_counter += 1\n                    patience = getattr(self.cfg, \'patience\', 30)\n                    if self._patience_counter >= patience:\n                        print(\n                            f"\\n🛑 [EARLY STOPPING TRIGGERED] Kích hoạt dừng sớm sau {patience} lần đánh giá "\n                            f"không cải thiện {getattr(self.cfg, \'which4best\', \'NDCG@20\')} (Best Epoch: {self._best_ep}).\\n"\n                        )\n                        self.cfg.epochs = epoch + 1\n            except Exception as e:\n                pass\n\n\n# ═════════════════════════════════════════════════════════════════════════════\n# Main Execution Entry Point\n# ═════════════════════════════════════════════════════════════════════════════\ndef main():\n    # Auto-bridge dataset for FreeRec:\n    processed_dir = os.path.join(cfg.root, "Processed", cfg.dataset)\n    if os.path.islink(processed_dir) and not os.path.exists(processed_dir):\n        try:\n            os.unlink(processed_dir)\n        except Exception:\n            pass\n\n    if not os.path.exists(processed_dir) or (os.path.isdir(processed_dir) and not os.listdir(processed_dir)):\n        script_dir = os.path.dirname(os.path.abspath(__file__)) if \'__file__\' in locals() else \'.\'\n        candidates = [\n            os.path.join(cfg.root, cfg.dataset),\n            os.path.join("/kaggle/data", cfg.dataset),\n            os.path.join("/kaggle/data/Processed", cfg.dataset),\n            os.path.join("/kaggle/working/STAIR/data", cfg.dataset),\n            os.path.join("/kaggle/working/STAIR-Enhanced/data", cfg.dataset),\n            os.path.join(script_dir, "data", cfg.dataset),\n            os.path.join("data", cfg.dataset),\n            os.path.join("data/Processed", cfg.dataset),\n        ]\n        for cand in candidates:\n            if os.path.exists(cand) and os.path.isdir(cand) and os.path.abspath(cand) != os.path.abspath(processed_dir) and len(os.listdir(cand)) > 0:\n                os.makedirs(os.path.dirname(processed_dir), exist_ok=True)\n                try:\n                    os.symlink(cand, processed_dir)\n                    print(f"[DataSet] >>> Auto-bridged symlink: {cand} -> {processed_dir}")\n                except Exception:\n                    import shutil\n                    shutil.copytree(cand, processed_dir, dirs_exist_ok=True)\n                    print(f"[DataSet] >>> Auto-bridged copied: {cand} -> {processed_dir}")\n                break\n\n    # Robust dataset loading:\n    tasktag = getattr(cfg, \'tasktag\', None) or getattr(freerec.data.tags, \'MATCHING\', None)\n    if hasattr(freerec.data.datasets, \'RecDataSet\'):\n        freerec.data.datasets.RecDataSet.TASK = tasktag\n    if hasattr(freerec.data.datasets, \'base\') and hasattr(freerec.data.datasets.base, \'BaseSet\'):\n        freerec.data.datasets.base.BaseSet.TASK = tasktag\n\n    ds_cls = getattr(freerec.data.datasets, cfg.dataset, None)\n    if isinstance(ds_cls, type):\n        try:\n            dataset = ds_cls(root=cfg.root)\n        except Exception:\n            try:\n                from freerec.data.datasets.base import MatchingRecDataSet\n                dataset = MatchingRecDataSet(cfg.root, cfg.dataset, tasktag=tasktag)\n            except Exception:\n                dataset = freerec.data.datasets.RecDataSet(\n                    cfg.root, cfg.dataset, tasktag=tasktag\n                )\n    else:\n        try:\n            from freerec.data.datasets.base import MatchingRecDataSet\n            dataset = MatchingRecDataSet(cfg.root, cfg.dataset, tasktag=tasktag)\n        except Exception:\n            dataset = freerec.data.datasets.RecDataSet(\n                cfg.root, cfg.dataset, tasktag=tasktag\n            )\n\n    if not hasattr(dataset, \'TASK\') or dataset.TASK is None:\n        dataset.TASK = tasktag\n\n    model = STAIR_NE_NLGCL_v5_Plus_Model(dataset)\n\n    trainpipe = model.sure_trainpipe(cfg.batch_size)\n    validpipe = model.sure_validpipe(cfg.ranking)\n    testpipe  = model.sure_testpipe(cfg.ranking)\n\n    coach = CoachForSTAIR_NE_NLGCL_v5_Plus(\n        dataset=dataset,\n        trainpipe=trainpipe,\n        validpipe=validpipe,\n        testpipe=testpipe,\n        model=model,\n        cfg=cfg,\n    )\n\n    if torch.cuda.is_available():\n        torch.cuda.reset_peak_memory_stats()\n\n    coach.fit()\n\n    # FreeRec\'s coach.summary() (called automatically at the end of coach.fit())\n    # already restores the best model checkpoint (best.pt) and evaluates both VALID and TEST.\n    # We copy the best checkpoint to best_model.pth for compatibility:\n    save_dir = getattr(cfg, \'CHECKPOINT_PATH\', getattr(cfg, \'root_dir\', \'.\'))\n    best_pt_path = os.path.join(save_dir, getattr(cfg, \'BEST_FILENAME\', \'best.pt\'))\n    best_pth_path = os.path.join(save_dir, "best_model.pth")\n    if os.path.exists(best_pt_path) and not os.path.exists(best_pth_path):\n        try:\n            import shutil\n            shutil.copy2(best_pt_path, best_pth_path)\n            print(f"[Coach] >>> Đã lưu bản sao mô hình tối ưu: {best_pth_path}")\n        except Exception:\n            pass\n\n    print("\\n[Coach] >>> HOÀN TẤT HUẤN LUYỆN VÀ ĐÁNH GIÁ TỐI ƯU THÀNH CÔNG (Exit Code: 0)!")\n\n    if torch.cuda.is_available():\n        max_alloc_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)\n        max_res_mb   = torch.cuda.max_memory_reserved() / (1024 ** 2)\n        print("=" * 80)\n        print("[VRAM TELEMETRY — PYTORCH ALLOCATOR (AUTHOR PAPER METHOD)]")\n        print(f"  * Pure Tensor Peak (max_memory_allocated) : {max_alloc_mb:.2f} MB")\n        print(f"  * Peak Reserved Memory (max_memory_reserved): {max_res_mb:.2f} MB")\n        print("=" * 80)\n\n\nif __name__ == \'__main__\':\n    main()\n')
with open(os.path.join(STAIR_DIR, 'main.py'), 'w', encoding='utf-8') as f:
    f.write('\n\nfrom typing import Dict, Tuple, Optional\n\nimport torch, os, math\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport freerec\n\nfrom optimizers.Adam import AdamSEvo\nfrom optimizers.AdamW import AdamWSEvo\nfrom optimizers.utils import Smoother\n\nfreerec.declare(version=\'1.0.1\')\n\ncfg = freerec.parser.Parser()\ncfg.add_argument("--embedding-dim", type=int, default=64)\ncfg.add_argument("--num-layers", type=int, default=3, help="the number of layers for FSC/BSC")\n\ncfg.add_argument("--mfiles", type=str, default="textual_modality.pkl,visual_modality.pkl", help="the files saving modality")\ncfg.add_argument("--num-neighbors", type=str, default=\'5-1\', help="for kNN graph")\ncfg.add_argument("--gamma", type=float, default=0.2)\ncfg.add_argument("--patience", type=int, default=30, help="early stopping patience (default: 30)")\n\ncfg.set_defaults(\n    description="STAIR",\n    root="../../data",\n    dataset=\'Amazon2014Baby_550_MMRec\',\n    epochs=500,\n    batch_size=1024,\n    optimizer=\'adamwsevo\',\n    lr=1e-3,\n    weight_decay=0.1,\n    seed=1,\n    monitors=["Recall@10", "Recall@20", "NDCG@10", "NDCG@20"],\n    which4best="NDCG@20",\n)\ncfg.compile()\n\n\nif isinstance(cfg.mfiles, str):\n    cfg.mfiles = cfg.mfiles.split(\',\')\nif isinstance(cfg.num_neighbors, str):\n    cfg.num_neighbors = list(map(int, cfg.num_neighbors.split(\'-\'))) \n\n# beta3 here is the 1 - beta_j for BSC\ncfg.beta3 = (0.1 + 0.9 * (torch.arange(cfg.embedding_dim) / cfg.embedding_dim).pow(cfg.gamma)).to(cfg.device)\n\nclass STAIR(freerec.models.GenRecArch):\n\n    def __init__(\n        self, dataset: freerec.data.datasets.RecDataSet\n    ) -> None:\n        super().__init__(dataset)\n\n        self.num_layers = cfg.num_layers\n\n        self.User.add_module(\n            "embeddings", nn.Embedding(\n                self.User.count, cfg.embedding_dim\n            )\n        )\n\n        self.Item.add_module(\n            "embeddings", nn.Embedding(\n                self.Item.count, cfg.embedding_dim\n            )\n        )\n\n        self.register_buffer(\n            "Adj",\n            self.dataset.train().to_normalized_adj(\n                normalization=\'sym\'\n            )\n        )\n\n        self.reset_parameters()\n\n        self.prepare(dataset.path)\n\n        self.criterion = freerec.criterions.BPRLoss(reduction=\'mean\')\n\n    def reset_parameters(self):\n        for m in self.modules():\n            if isinstance(m, nn.Linear):\n                nn.init.kaiming_normal_(m.weight)\n                if m.bias is not None:\n                    nn.init.constant_(m.bias, 0.)\n            elif isinstance(m, nn.Embedding):\n                nn.init.normal_(m.weight, std=1.e-4)\n            elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):\n                nn.init.constant_(m.weight, 1.)\n                nn.init.constant_(m.bias, 0.)\n\n    def marked_params(self):\n        params = [\n            {\n                \'params\': self.User.parameters(),\n                \'smoother\': None\n            },\n            {\n                \'params\': self.Item.parameters(), \n                \'smoother\': Smoother(self.mAdj, beta=cfg.beta3, L=cfg.num_layers, aggr=\'neumann\')\n            },\n        ]\n        return params\n\n    def whitening(self, feats: torch.Tensor):\n        if not isinstance(feats, torch.Tensor):\n            feats = torch.tensor(feats, dtype=torch.float32)\n        else:\n            feats = feats.float()\n        feats = feats - feats.mean(0, keepdim=True)\n        feats, _, _ = torch.linalg.svd(feats, full_matrices=False)\n        if feats.size(1) < cfg.embedding_dim:\n            reps = math.ceil(cfg.embedding_dim / feats.size(1))\n            scale = math.sqrt(feats.size(1) / cfg.embedding_dim)\n            feats = (feats.repeat(1, reps)[:, :cfg.embedding_dim]) * scale\n        else:\n            feats = feats[:, :cfg.embedding_dim]\n        return feats * math.sqrt(self.Item.count / cfg.embedding_dim)\n\n    def get_knn_graph(self, features: torch.Tensor, k: int = 5):\n        r"""\n        Compute the kNN graph.\n        """\n        if not isinstance(features, torch.Tensor):\n            features = torch.tensor(features, dtype=torch.float32)\n        else:\n            features = features.float()\n        features = F.normalize(features, dim=-1) # (N, D)\n        sim = features @ features.t() # (N, N)\n        sim.fill_diagonal_(-10.)\n        edge_index, _ = freerec.graph.get_knn_graph(\n            sim, k, symmetric=False\n        )\n        return edge_index\n\n    def prepare(self, path: str):\n        from freerec.utils import import_pickle\n\n        mfeats = []\n        for mfile in cfg.mfiles:\n            mpath = os.path.join(path, mfile)\n            if not os.path.exists(mpath):\n                # Fallback search across common data locations\n                for cand in [\n                    os.path.join(cfg.root, cfg.dataset, mfile),\n                    os.path.join("/kaggle/data", cfg.dataset, mfile),\n                    os.path.join("/kaggle/working/STAIR/data", cfg.dataset, mfile),\n                    os.path.join("/kaggle/working/STAIR-Enhanced/data", cfg.dataset, mfile),\n                    os.path.join("data", cfg.dataset, mfile),\n                ]:\n                    if os.path.exists(cand):\n                        mpath = cand\n                        break\n            mfeats.append(import_pickle(mpath))\n\n        edge_index = torch.cat(\n            [self.get_knn_graph(feats, k) for feats, k in zip(mfeats, cfg.num_neighbors)],\n            dim=1\n        )\n        edge_weight = torch.ones_like(edge_index[0], dtype=torch.float)\n        edge_index, edge_weight = freerec.graph.coalesce(\n            edge_index, edge_weight, reduce=\'sum\'\n        )\n        edge_index, edge_weight = freerec.graph.to_undirected(\n            edge_index, edge_weight, reduce=\'max\'\n        )\n        edge_index, edge_weight = freerec.graph.to_normalized(\n            edge_index, edge_weight,\n            normalization=\'sym\'\n        )\n        mAdj = torch.sparse_coo_tensor(\n            edge_index, edge_weight,\n            size=(self.Item.count, self.Item.count)\n        )\n        self.register_buffer(\n            \'mAdj\',\n            mAdj.to_sparse_csr()\n        )\n\n        # MI\n        mfeats = [self.whitening(mfeat) * k for mfeat, k in zip(mfeats, cfg.num_neighbors)]\n        mfeats = sum(mfeats).div(sum(cfg.num_neighbors))\n        self.Item.embeddings.weight.data.copy_(mfeats)\n\n        edge_index = self.dataset.train().to_bigraph(edge_type=\'u2i\')[\'u2i\'].edge_index\n        edge_index, edge_weight = freerec.graph.to_normalized(edge_index, normalization=\'left\')\n        R = torch.sparse_coo_tensor(\n            edge_index, edge_weight, size=(self.User.count, self.Item.count)\n        ).to_sparse_csr()\n\n        self.User.embeddings.weight.data.copy_(R @ mfeats)\n\n    def sure_trainpipe(self, batch_size: int):\n        return self.dataset.train().shuffled_pairs_source(\n        ).gen_train_sampling_neg_(\n            num_negatives=1\n        ).batch_(batch_size).tensor_()\n\n    def encode(self) -> Tuple[torch.Tensor, torch.Tensor]:\n        allEmbds = torch.cat(\n            (self.User.embeddings.weight, self.Item.embeddings.weight), dim=0\n        ) # (N, D)\n\n        features = allEmbds\n        smoothed = allEmbds\n        \n        # FSC\n        beta = 1 - cfg.beta3\n        norm_correction = 1 - beta ** (self.num_layers + 1)\n        for _ in range(self.num_layers):\n            features = self.Adj @ features * beta\n            smoothed = smoothed + features\n        avgEmbds = smoothed.mul(1 - beta).div(norm_correction)\n        userEmbds, itemEmbds = torch.split(\n            avgEmbds, (self.User.count, self.Item.count)\n        )\n        return userEmbds, itemEmbds\n\n    def fit(self, data: Dict[freerec.data.fields.Field, torch.Tensor]):\n        userEmbds, itemEmbds = self.encode()\n        users, positives, negatives = data[self.User], data[self.Item], data[self.INeg]\n        userEmbds = userEmbds[users] # (B, 1, D)\n        iposEmbds = itemEmbds[positives] # (B, 1, D)\n        inegEmbds = itemEmbds[negatives] # (B, K, D)\n\n        rec_loss = self.criterion(\n            torch.einsum("BKD,BKD->BK", userEmbds, iposEmbds),\n            torch.einsum("BKD,BKD->BK", userEmbds, inegEmbds)\n        )\n        return rec_loss\n\n    def reset_ranking_buffers(self):\n        """This method will be executed before evaluation."""\n        userEmbds, itemEmbds = self.encode()\n        self.ranking_buffer = dict()\n        self.ranking_buffer[self.User] = userEmbds.detach().clone()\n        self.ranking_buffer[self.Item] = itemEmbds.detach().clone()\n\n    def recommend_from_full(self, data: Dict[freerec.data.fields.Field, torch.Tensor]):\n        userEmbds = self.ranking_buffer[self.User][data[self.User]] # (B, 1, D)\n        itemEmbds = self.ranking_buffer[self.Item]\n        return torch.einsum("BKD,ND->BN", userEmbds, itemEmbds)\n\n    def recommend_from_pool(self, data: Dict[freerec.data.fields.Field, torch.Tensor]):\n        userEmbds = self.ranking_buffer[self.User][data[self.User]] # (B, 1, D)\n        itemEmbds = self.ranking_buffer[self.Item][data[self.IUnseen]] # (B, 101, D)\n        return torch.einsum("BKD,BKD->BK", userEmbds, itemEmbds)\n\n\nclass CoachForSTAIR(freerec.launcher.Coach):\n\n    def set_optimizer(self):\n        if self.cfg.optimizer.lower() == \'sgd\':\n            self.optimizer = torch.optim.SGD(\n                self.model.marked_params(), lr=self.cfg.lr, \n                momentum=self.cfg.momentum,\n                nesterov=self.cfg.nesterov,\n                weight_decay=self.cfg.weight_decay\n            )\n        elif self.cfg.optimizer.lower() == \'adam\':\n            self.optimizer = torch.optim.Adam(\n                self.model.marked_params(), lr=self.cfg.lr,\n                betas=(self.cfg.beta1, self.cfg.beta2),\n                weight_decay=self.cfg.weight_decay\n            )\n        elif self.cfg.optimizer.lower() == \'adamw\':\n            self.optimizer = torch.optim.AdamW(\n                self.model.marked_params(), lr=self.cfg.lr,\n                betas=(self.cfg.beta1, self.cfg.beta2),\n                weight_decay=self.cfg.weight_decay\n            )\n        elif self.cfg.optimizer.lower() == \'adamsevo\':\n            self.optimizer = AdamSEvo(\n                self.model.marked_params(), lr=self.cfg.lr,\n                betas=(self.cfg.beta1, self.cfg.beta2),\n                weight_decay=self.cfg.weight_decay\n            )\n        elif self.cfg.optimizer.lower() == \'adamwsevo\':\n            self.optimizer = AdamWSEvo(\n                self.model.marked_params(), lr=self.cfg.lr,\n                betas=(self.cfg.beta1, self.cfg.beta2),\n                weight_decay=self.cfg.weight_decay\n            )\n        else:\n            raise NotImplementedError(\n                f"Unexpected optimizer {self.cfg.optimizer} ..."\n            )\n\n    def train_per_epoch(self, epoch: int):\n        for data in self.dataloader:\n            data = self.dict_to_device(data)\n            loss = self.model(data)\n\n            self.optimizer.zero_grad()\n            loss.backward()\n            self.optimizer.step()\n            \n            self.monitor(\n                loss.item(), \n                n=len(data[self.User]), reduction="mean", \n                mode=\'train\', pool=[\'LOSS\']\n            )\n\n    def evaluate(self, epoch: int = 0, mode: str = \'valid\'):\n        super().evaluate(epoch, mode=mode)\n        if mode == \'valid\':\n            try:\n                if not hasattr(self, \'_patience_counter\'):\n                    self._patience_counter = 0\n                    self._best_ep = getattr(self, \'best_epoch\', 0)\n\n                cur_best_ep = getattr(self, \'best_epoch\', 0)\n                if cur_best_ep == epoch:\n                    self._patience_counter = 0\n                    self._best_ep = epoch\n                else:\n                    self._patience_counter += 1\n                    patience = getattr(self.cfg, \'patience\', 30)\n                    if self._patience_counter >= patience:\n                        print(\n                            f"\\n🛑 [EARLY STOPPING TRIGGERED] Kích hoạt dừng sớm sau {patience} lần đánh giá "\n                            f"không cải thiện {getattr(self.cfg, \'which4best\', \'NDCG@20\')} (Best Epoch: {self._best_ep}).\\n"\n                        )\n                        self.cfg.epochs = epoch + 1\n            except Exception as e:\n                pass\n\n\ndef main():\n\n    # Robust auto-bridge for FreeRec:\n    processed_dir = os.path.join(cfg.root, "Processed", cfg.dataset)\n    if os.path.islink(processed_dir) and not os.path.exists(processed_dir):\n        try:\n            os.unlink(processed_dir)\n        except Exception:\n            pass\n\n    if not os.path.exists(processed_dir) or (os.path.isdir(processed_dir) and not os.listdir(processed_dir)):\n        script_dir = os.path.dirname(os.path.abspath(__file__)) if \'__file__\' in locals() else \'.\'\n        candidates = [\n            os.path.join(cfg.root, cfg.dataset),\n            os.path.join("/kaggle/data", cfg.dataset),\n            os.path.join("/kaggle/data/Processed", cfg.dataset),\n            os.path.join("/kaggle/working/STAIR/data", cfg.dataset),\n            os.path.join("/kaggle/working/STAIR-Enhanced/data", cfg.dataset),\n            os.path.join(script_dir, "data", cfg.dataset),\n            os.path.join("data", cfg.dataset),\n        ]\n        for cand in candidates:\n            if os.path.exists(cand) and os.path.isdir(cand) and os.path.abspath(cand) != os.path.abspath(processed_dir) and len(os.listdir(cand)) > 0:\n                os.makedirs(os.path.dirname(processed_dir), exist_ok=True)\n                try:\n                    os.symlink(cand, processed_dir)\n                    print(f"[DataSet] >>> Auto-bridged symlink: {cand} -> {processed_dir}")\n                except Exception:\n                    import shutil\n                    shutil.copytree(cand, processed_dir, dirs_exist_ok=True)\n                    print(f"[DataSet] >>> Auto-bridged copied: {cand} -> {processed_dir}")\n                break\n\n    # Robust dataset loading:\n    # 1. freerec.data.datasets contains a submodule named \'tiktok\', so getattr(...) returns a module\n    #    rather than a class when cfg.dataset == \'tiktok\'.\n    # 2. FreeRec\'s ValidSampler/TestSampler requires dataset.TASK is MATCHING. We define it on\n    #    RecDataSet class, BaseSet class, and pass tasktag to ensure it is always present.\n    tasktag = getattr(cfg, \'tasktag\', None) or getattr(freerec.data.tags, \'MATCHING\', None)\n    if hasattr(freerec.data.datasets, \'RecDataSet\'):\n        freerec.data.datasets.RecDataSet.TASK = tasktag\n    if hasattr(freerec.data.datasets, \'base\') and hasattr(freerec.data.datasets.base, \'BaseSet\'):\n        freerec.data.datasets.base.BaseSet.TASK = tasktag\n\n    ds_cls = getattr(freerec.data.datasets, cfg.dataset, None)\n    if isinstance(ds_cls, type):\n        try:\n            dataset = ds_cls(root=cfg.root)\n        except Exception:\n            try:\n                from freerec.data.datasets.base import MatchingRecDataSet\n                dataset = MatchingRecDataSet(cfg.root, cfg.dataset, tasktag=tasktag)\n            except Exception:\n                dataset = freerec.data.datasets.RecDataSet(\n                    cfg.root, cfg.dataset, tasktag=tasktag\n                )\n    else:\n        try:\n            from freerec.data.datasets.base import MatchingRecDataSet\n            dataset = MatchingRecDataSet(cfg.root, cfg.dataset, tasktag=tasktag)\n        except Exception:\n            dataset = freerec.data.datasets.RecDataSet(\n                cfg.root, cfg.dataset, tasktag=tasktag\n            )\n\n    # Ensure TASK attribute is always attached to dataset instance\n    if not hasattr(dataset, \'TASK\') or dataset.TASK is None:\n        dataset.TASK = tasktag\n\n    model = STAIR(dataset)\n\n    trainpipe = model.sure_trainpipe(cfg.batch_size)\n    validpipe = model.sure_validpipe(cfg.ranking)\n    testpipe = model.sure_testpipe(cfg.ranking)\n\n    coach = CoachForSTAIR(\n        dataset=dataset,\n        trainpipe=trainpipe,\n        validpipe=validpipe,\n        testpipe=testpipe,\n        model=model,\n        cfg=cfg\n    )\n    coach.fit()\n\n    save_dir = getattr(cfg, \'CHECKPOINT_PATH\', getattr(cfg, \'root_dir\', \'.\'))\n    best_pt_path = os.path.join(save_dir, getattr(cfg, \'BEST_FILENAME\', \'best.pt\'))\n    best_pth_path = os.path.join(save_dir, "best_model.pth")\n    if os.path.exists(best_pt_path) and not os.path.exists(best_pth_path):\n        try:\n            import shutil\n            shutil.copy2(best_pt_path, best_pth_path)\n            print(f"[Coach] >>> Đã lưu bản sao mô hình tối ưu: {best_pth_path}")\n        except Exception:\n            pass\n\n    print("\\n[Coach] >>> HOÀN TẤT HUẤN LUYỆN VÀ ĐÁNH GIÁ TỐI ƯU THÀNH CÔNG (Exit Code: 0)!")\n\n\nif __name__ == "__main__":\n    main()')
with open(os.path.join(STAIR_DIR, 'configs', 'Amazon2014Clothing_550_MMRec.yaml'), 'w', encoding='utf-8') as f:
    f.write("root: data\ndataset: Amazon2014Clothing_550_MMRec\n\nembedding_dim: 64\nnum_layers: 3\n\nepochs: 500\nbatch_size: 2048\noptimizer: adamwsevo\nlr: 1.e-3\nweight_decay: 0.1\npatience: 30\neval_freq: 5\n\ngamma: 0.2\nmfiles: textual_modality.pkl,visual_modality.pkl\nnum_neighbors: '5-1'\n\nmonitors: [LOSS, Recall@1, Recall@10, Recall@20, NDCG@10, NDCG@20]\nwhich4best: NDCG@20\n")

# Xóa cache module để kernel luôn nạp phiên bản mới nhất từ đĩa
for mod_name in list(sys.modules.keys()):
    if 'stair_ne_nlgcl' in mod_name or 'stair_breakthrough' in mod_name:
        sys.modules.pop(mod_name, None)

# 2. Cài đặt các gói phụ thuộc bắt buộc
print("📦 Cài đặt dependencies (torchdata, torch_geometric, freerec, nvidia-ml-py, prettytable, scipy)...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchdata==0.7.1'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch_geometric', 'freerec==0.8.5', 'nvidia-ml-py', 'prettytable', 'matplotlib', 'pyyaml', 'seaborn', 'scipy', 'pandas'
], check=True)

# 3. Kaggle TorchData compatibility shims cho FreeRec (PyTorch 2.x & Python 3.10+)
import types
import torch.utils.data

try:
    import torchdata
    import torchdata.datapipes as dp
except Exception:
    dp = None

if dp is None or 'torchdata.datapipes' not in sys.modules:
    if 'torchdata' not in sys.modules:
        td = types.ModuleType('torchdata')
        sys.modules['torchdata'] = td
    else:
        td = sys.modules['torchdata']
    dp = types.ModuleType('torchdata.datapipes')
    td.datapipes = dp
    sys.modules['torchdata.datapipes'] = dp

if not hasattr(dp, 'iter'):
    iter_mod = types.ModuleType('torchdata.datapipes.iter')
    dp.iter = iter_mod
    sys.modules['torchdata.datapipes.iter'] = iter_mod
if not hasattr(dp.iter, 'IterDataPipe'):
    class IterDataPipe(torch.utils.data.IterableDataset):
        def __iter__(self): return iter([])
    dp.iter.IterDataPipe = IterDataPipe

if not hasattr(dp, 'map'):
    map_mod = types.ModuleType('torchdata.datapipes.map')
    dp.map = map_mod
    sys.modules['torchdata.datapipes.map'] = map_mod
if not hasattr(dp.map, 'MapDataPipe'):
    class MapDataPipe(torch.utils.data.Dataset):
        def __getitem__(self, idx): return None
    dp.map.MapDataPipe = MapDataPipe

def functional_datapipe(name, enable_df_api=False):
    def decorator(cls):
        def method(*args, **kwargs):
            return cls(*args, **kwargs)
        if hasattr(dp, 'iter') and hasattr(dp.iter, 'IterDataPipe'):
            setattr(dp.iter.IterDataPipe, name, method)
        if hasattr(dp, 'map') and hasattr(dp.map, 'MapDataPipe'):
            setattr(dp.map.MapDataPipe, name, method)
        try:
            if hasattr(torch.utils.data, 'IterDataPipe'):
                setattr(torch.utils.data.IterDataPipe, name, method)
            if hasattr(torch.utils.data, 'MapDataPipe'):
                setattr(torch.utils.data.MapDataPipe, name, method)
        except Exception:
            pass
        return cls
    return decorator

dp.functional_datapipe = functional_datapipe
print("✅ Môi trường STAIR-Enhanced & Dependencies đã sẵn sàng cho Clothing!")


In [ ]:
# Cell 2: Tự Động Quét, Giải Nén & Chuẩn Hóa Dữ Liệu Clothing Sang Chuẩn FreeRec/STAIR
import os, pickle, shutil, time, zipfile
import numpy as np
import pandas as pd
import torch

DATA_ROOT = '/kaggle/data'
PROCESSED_ROOT = os.path.join(DATA_ROOT, 'Processed')
LOCAL_DATA = os.path.join(os.getcwd(), 'data')
LOCAL_PROCESSED = os.path.join(LOCAL_DATA, 'Processed')

CANONICAL_NAMES = ['Amazon2014Clothing_550_MMRec', 'clothing']
REQUIRED_PROCESSED_FILES = [
    'train.txt', 'valid.txt', 'test.txt',
    'visual_modality.pkl', 'textual_modality.pkl'
]

def bridge_directories(src_path):
    for folder in CANONICAL_NAMES:
        for base_dir in [PROCESSED_ROOT, DATA_ROOT, LOCAL_DATA, LOCAL_PROCESSED]:
            dst = os.path.join(base_dir, folder)
            if os.path.abspath(src_path) != os.path.abspath(dst):
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                if os.path.islink(dst) or os.path.exists(dst):
                    try:
                        if os.path.islink(dst):
                            os.unlink(dst)
                        else:
                            shutil.rmtree(dst, ignore_errors=True)
                    except Exception:
                        pass
                try:
                    os.symlink(src_path, dst)
                except Exception:
                    shutil.copytree(src_path, dst, dirs_exist_ok=True)

def convert_raw_clothing(inter_path, text_npy_path, img_npy_path, dst_dir):
    print(f"🔄 Đang chuyển đổi dữ liệu Clothing từ {inter_path} sang chuẩn FreeRec ({dst_dir})...")
    os.makedirs(dst_dir, exist_ok=True)

    df = pd.read_csv(inter_path, sep='\t')
    print(f"  • Tổng số tương tác: {len(df):,}")
    print(f"  • Số người dùng (Users): {df['userID'].nunique():,} | Số mặt hàng (Items): {df['itemID'].nunique():,}")

    splits = [
        ('train.txt', df[df['x_label'] == 0]),
        ('valid.txt', df[df['x_label'] == 1]),
        ('test.txt',  df[df['x_label'] == 2]),
    ]
    for fname, sub_df in splits:
        out_p = os.path.join(dst_dir, fname)
        with open(out_p, 'w', encoding='utf-8') as f:
            f.write("USER\tITEM\tTIMESTAMP\n")
            for u, i, t in zip(sub_df['userID'], sub_df['itemID'], sub_df['timestamp']):
                f.write(f"{u}\t{i}\t{t}\n")
        print(f"  [Đã sinh] {fname}: {len(sub_df):,} tương tác -> {out_p}")

    # Xử lý text features
    if os.path.exists(text_npy_path):
        text_feat = np.load(text_npy_path)
        text_tensor = torch.from_numpy(text_feat.astype(np.float32))
        with open(os.path.join(dst_dir, 'textual_modality.pkl'), 'wb') as f:
            pickle.dump(text_tensor, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"  [Đã sinh] textual_modality.pkl: {text_tensor.shape} ({text_tensor.dtype})")

    # Xử lý image features
    if os.path.exists(img_npy_path):
        img_feat = np.load(img_npy_path)
        img_tensor = torch.from_numpy(img_feat.astype(np.float32))
        with open(os.path.join(dst_dir, 'visual_modality.pkl'), 'wb') as f:
            pickle.dump(img_tensor, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"  [Đã sinh] visual_modality.pkl: {img_tensor.shape} ({img_tensor.dtype})")

    print("✅ Hoàn tất chuyển đổi Clothing sang chuẩn FreeRec/STAIR!")

def scan_and_prepare_clothing():
    target_processed = os.path.join(PROCESSED_ROOT, 'Amazon2014Clothing_550_MMRec')

    # 1. Kiểm tra nếu đã có sẵn dạng processed
    for folder in CANONICAL_NAMES:
        for cand in [os.path.join(PROCESSED_ROOT, folder), os.path.join(DATA_ROOT, folder), os.path.join(LOCAL_PROCESSED, folder)]:
            if os.path.exists(cand) and all(os.path.exists(os.path.join(cand, f)) for f in REQUIRED_PROCESSED_FILES):
                print(f"✅ Dữ liệu Clothing đã tồn tại sẵn tại: {cand}")
                bridge_directories(cand)
                return cand

    # 2. Quét /kaggle/input tìm kiếm Clothing
    print("🔍 Đang quét /kaggle/input để tìm kiếm tập dữ liệu Clothing...")
    input_base = '/kaggle/input'
    raw_inter = None
    raw_text = None
    raw_img = None

    for attempt in range(12):
        if os.path.exists(input_base):
            for root, dirs, files in os.walk(input_base):
                if all(rf in files for rf in REQUIRED_PROCESSED_FILES):
                    print(f"  [TÌM THẤY DỮ LIỆU ĐÃ XỬ LÝ] {root}")
                    os.makedirs(target_processed, exist_ok=True)
                    for f in files:
                        shutil.copy2(os.path.join(root, f), os.path.join(target_processed, f))
                    bridge_directories(target_processed)
                    return target_processed

                if 'clothing.inter' in files:
                    raw_inter = os.path.join(root, 'clothing.inter')
                    raw_text = os.path.join(root, 'text_feat.npy')
                    raw_img = os.path.join(root, 'image_feat.npy')
                    break
            if raw_inter:
                break

        # Kiểm tra file nén trong repository
        for zname in ['clothing.zip', 'Amazon2014Clothing_550_MMRec.zip']:
            repo_zip = os.path.join(LOCAL_DATA, zname)
            if not raw_inter and os.path.exists(repo_zip):
                print(f"📦 Phát hiện tệp nén {repo_zip}. Đang giải nén...")
                ext_dir = os.path.join(LOCAL_DATA, 'raw_clothing')
                with zipfile.ZipFile(repo_zip, 'r') as zf:
                    zf.extractall(ext_dir)
                for root, dirs, files in os.walk(ext_dir):
                    if 'clothing.inter' in files:
                        raw_inter = os.path.join(root, 'clothing.inter')
                        raw_text = os.path.join(root, 'text_feat.npy')
                        raw_img = os.path.join(root, 'image_feat.npy')
                        break
                    if all(rf in files for rf in REQUIRED_PROCESSED_FILES):
                        bridge_directories(root)
                        return root

        if raw_inter:
            break
        print(f"⏳ Dataset đang được tải về từ Kaggle... Đợi 5s (lần {attempt+1}/12)...")
        time.sleep(5)

    if raw_inter and os.path.exists(raw_inter):
        convert_raw_clothing(raw_inter, raw_text, raw_img, target_processed)
        bridge_directories(target_processed)
        return target_processed
    else:
        print("❌ Không tìm thấy tập dữ liệu Clothing trong /kaggle/input hoặc trong repository!")
        return None

prepared_clothing_dir = scan_and_prepare_clothing()
print("=" * 75)
p_dir = os.path.join(PROCESSED_ROOT, 'Amazon2014Clothing_550_MMRec')
ready_files = [f for f in REQUIRED_PROCESSED_FILES if os.path.exists(os.path.join(p_dir, f))]
status = f"✅ {len(ready_files)}/{len(REQUIRED_PROCESSED_FILES)} tệp chuẩn sẵn sàng" if len(ready_files) == len(REQUIRED_PROCESSED_FILES) else "❌ THIẾU"
print(f"TỔNG KẾT DỮ LIỆU CLOTHING: {status} trong {p_dir}")
for rf in REQUIRED_PROCESSED_FILES:
    fp = os.path.join(p_dir, rf)
    if os.path.exists(fp):
        sz = os.path.getsize(fp) / (1024 * 1024)
        print(f"  • {rf:25s}: {sz:.2f} MB")
print("=" * 75)


In [ ]:
# Cell 3: Runner Engine Huấn Luyện & Trích Xuất Metrics Tối Ưu (Zero-OOM, Auto-Parse)
import subprocess, sys, os, time, re, threading

try:
    import pynvml
    pynvml.nvmlInit()
    HAS_NVML = True
except Exception:
    HAS_NVML = False

def get_gpu_memory_used():
    if not HAS_NVML:
        return 0.0
    try:
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        return info.used / (1024 ** 2)
    except Exception:
        return 0.0

TRACKED_METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

def extract_best_test(log_path):
    if not os.path.exists(log_path):
        return None, {}

    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
        lines = content.splitlines()

    best_epoch = None
    best_metrics = {}

    # 1. Tìm best_epoch từ log FreeRec
    m_load = re.findall(r'Load best model @Epoch\s+(\d+)', content)
    if m_load:
        best_epoch = int(m_load[-1])
    else:
        m_tbl = re.findall(r'valid\s+NDCG@20\s+[0-9.]+\s+(\d+)', content)
        if m_tbl:
            best_epoch = int(m_tbl[-1])
        else:
            m_new = re.findall(r'NEW BEST MODEL @Epoch\s+(\d+)', content)
            if m_new:
                best_epoch = int(m_new[-1])

    # 2. Tìm metrics TEST tại best_epoch hoặc dòng TEST summary cuối cùng
    test_lines = [l for l in lines if 'TEST @Epoch:' in l and '||' in l]
    target_line = None
    if best_epoch is not None:
        for tl in reversed(test_lines):
            if f"TEST @Epoch: {best_epoch}" in tl or f"TEST @Epoch:{best_epoch}" in tl:
                target_line = tl
                break

    if not target_line and test_lines:
        target_line = test_lines[-1]
        if best_epoch is None:
            m_ep = re.search(r'TEST @Epoch:\s*(\d+)', target_line)
            if m_ep:
                best_epoch = int(m_ep.group(1))

    if target_line:
        for metric in TRACKED_METRICS:
            m = re.search(rf'{re.escape(metric)}\s+(?:Avg:\s*|:\s*)([0-9.]+)', target_line, re.IGNORECASE)
            if m:
                best_metrics[metric] = float(m.group(1))

    # 3. Fallback quét bảng Mode Metric Best @Epoch
    if len(best_metrics) < 4:
        for line in lines:
            for metric in TRACKED_METRICS:
                if metric not in best_metrics:
                    m = re.search(rf'test\s+{re.escape(metric)}\s+([0-9.]+)', line, re.IGNORECASE)
                    if m:
                        best_metrics[metric] = float(m.group(1))

    return best_epoch, best_metrics

def run_stair_baseline(
    key, yaml_cfg, data_root, log_path,
    epochs=500,
    embedding_dim=256,
    batch_size=2048,
    patience=30,
    mfiles='textual_modality.pkl,visual_modality.pkl',
    num_neighbors='5-1',
    **kwargs
):
    print('=' * 85)
    print(f'🚀 BẮT ĐẦU HUẤN LUYỆN: {key.upper()} | METHOD: [STAIR_BASELINE] | DIM: [{embedding_dim}] | EPOCHS: [{epochs}]')
    print(f'  * Dataset Key         : {key}')
    print(f'  * Method              : STAIR Baseline')
    print(f'  * Embedding Dim       : {embedding_dim} (Không gian 256D đối chuẩn)')
    print(f'  * Modal kNN Neighbors : {num_neighbors} (Text: 5, Vision: 1)')
    print(f'  * Batch Size          : {batch_size}')
    print(f'  * Max Epochs          : {epochs}')
    print(f'  * Early Stop Patience : {patience} eps')
    print(f'  * Target Metric       : NDCG@20 (N@20)')
    print(f'  * Log Path            : {log_path}')
    print('=' * 85)

    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    t0 = time.time()
    runner_py = '/kaggle/working/STAIR-Enhanced/main.py'
    if not os.path.exists(runner_py):
        runner_py = 'main.py'

    cmd = [
        sys.executable, runner_py,
        '--config',           yaml_cfg,
        '--root',             data_root,
        '--epochs',           str(epochs),
        '--embedding-dim',    str(embedding_dim),
        '--batch-size',       str(batch_size),
        '--patience',         str(patience),
        '--mfiles',           mfiles,
        '--num-neighbors',    num_neighbors,
    ]

    with open(log_path, 'w', encoding='utf-8') as f:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        for line in proc.stdout:
            print(line, end='', flush=True)
            f.write(line)
            f.flush()
        proc.wait()

    elapsed = time.time() - t0
    print('=' * 85)
    if proc.returncode != 0:
        print(f'⚠️ [CẢNH BÁO] STAIR Baseline ({embedding_dim}D) kết thúc với mã {proc.returncode}.')
    else:
        print(f'✅ [HOÀN TẤT] STAIR Baseline ({embedding_dim}D) thành công trong {elapsed/60:.2f} phút ({elapsed:.1f}s)!')

    best_ep, metrics = extract_best_test(log_path)
    print(f'  * Checkpoint tối ưu : Epoch {best_ep}')
    for m, val in metrics.items():
        print(f'  * {m:12s}: {val:.4f}')
    print('=' * 85)
    return best_ep, metrics

def run_training_v5_plus(
    key, yaml_cfg, data_root, log_path,
    method='dcd_gated',
    epochs=500,
    embedding_dim=256,
    batch_size=2048,
    lr_warmup_epochs=15,
    min_lr=1e-6,
    patience=30,
    tau=0.20, alpha_dir=0.50, eps=0.08, tau_thresh=0.85,
    lambda_cl=0.010, gamma_h=0.15, warmup_epochs=50,
    mfiles='textual_modality.pkl,visual_modality.pkl',
    num_neighbors='5-1',
    **kwargs
):
    print('=' * 85)
    print(f'🚀 BẮT ĐẦU HUẤN LUYỆN: {key.upper()} | METHOD: [{method.upper()}] | DIM: [{embedding_dim}] | EPOCHS: [{epochs}]')
    print(f'  * Dataset Key         : {key}')
    print(f'  * Method              : {method}')
    print(f'  * Embedding Dim       : {embedding_dim} (Không gian 256D đối chuẩn)')
    print(f'  * Modal kNN Neighbors : {num_neighbors} (Text: 5, Vision: 1)')
    print(f'  * Batch Size          : {batch_size}')
    print(f'  * Max Epochs          : {epochs}')
    print(f'  * LR Cosine Warmup    : {lr_warmup_epochs} eps -> Cosine Decay (min_lr: {min_lr})')
    print(f'  * Early Stop Patience : {patience} eps')
    print(f'  * Target Metric       : NDCG@20 (N@20)')
    print(f'  * Log Path            : {log_path}')
    print('=' * 85)

    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    t0 = time.time()
    runner_py = '/kaggle/working/STAIR-Enhanced/main_stair_ne_nlgcl_v5_plus.py'
    if not os.path.exists(runner_py):
        runner_py = 'main_stair_ne_nlgcl_v5_plus.py'

    cmd = [
        sys.executable, runner_py,
        '--config',           yaml_cfg,
        '--root',             data_root,
        '--method',           method,
        '--epochs',           str(epochs),
        '--embedding-dim',    str(embedding_dim),
        '--batch-size',       str(batch_size),
        '--lr-warmup-epochs', str(lr_warmup_epochs),
        '--min-lr',           str(min_lr),
        '--patience',         str(patience),
        '--tau',              str(tau),
        '--alpha-dir',        str(alpha_dir),
        '--eps',              str(eps),
        '--tau-thresh',       str(tau_thresh),
        '--lambda-cl',        str(lambda_cl),
        '--gamma-h',          str(gamma_h),
        '--warmup-epochs',    str(warmup_epochs),
        '--mfiles',           mfiles,
        '--num-neighbors',    num_neighbors,
    ]

    with open(log_path, 'w', encoding='utf-8') as f:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        for line in proc.stdout:
            print(line, end='', flush=True)
            f.write(line)
            f.flush()
        proc.wait()

    elapsed = time.time() - t0
    print('=' * 85)
    if proc.returncode != 0:
        print(f'⚠️ [CẢNH BÁO] Quá trình huấn luyện {key.upper()} ({method}) kết thúc (Mã: {proc.returncode}).')
    else:
        print(f'✅ [HOÀN TẤT] Huấn luyện {key.upper()} ({method}) thành công trong {elapsed/60:.2f} phút ({elapsed:.1f}s)!')

    best_ep, metrics = extract_best_test(log_path)
    print(f'  * Checkpoint tối ưu : Epoch {best_ep}')
    for m, val in metrics.items():
        print(f'  * {m:12s}: {val:.4f}')
    print('=' * 85)
    return best_ep, metrics


## 🏋️ PHẦN 1: Huấn luyện STAIR Baseline trên Clothing ở không gian 256 chiều (256D)
Chạy mô hình **STAIR Baseline gốc** khi mở rộng dung lượng biểu diễn lên **256 chiều** (`embedding_dim: 256`):
- Giữ nguyên toàn bộ cấu hình chuẩn của tác giả: $L=3$, $k_t=5, k_v=1$, tối ưu AdamW (`lr: 1e-3`, `weight_decay: 0.1`).
- Thử nghiệm này giúp trả lời câu hỏi khoa học quan trọng:
  *Tăng số chiều tự thân lên 256D có giúp Baseline cải thiện không, hay bị quá khớp (overfitting) do đồ thị Clothing quá thưa (99.97%)?*

In [ ]:
# Cell 5: Huấn luyện STAIR Baseline (256D, kNN 5-1, Batch 2048)
DATA_ROOT = '/kaggle/data'
YAML_PATH = '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Clothing_550_MMRec.yaml'
LOG_BASELINE_256D = '/kaggle/working/logs/baseline/clothing_stair_baseline_dim256.log'

run_stair_baseline(
    key='Amazon2014Clothing_550_MMRec',
    yaml_cfg=YAML_PATH,
    data_root=DATA_ROOT,
    log_path=LOG_BASELINE_256D,
    epochs=500,
    embedding_dim=256,
    batch_size=2048,
    patience=30,
    mfiles='textual_modality.pkl,visual_modality.pkl',
    num_neighbors='5-1',
)


## 🚀 PHẦN 2: Huấn luyện STAIR DCD-Gated trên Clothing ở không gian 256 chiều (256D)
Chạy phương pháp đột phá **STAIR DCD-Gated** với dung lượng **256 chiều** (`embedding_dim: 256`):
- Làm dày đồ thị với cạnh ảo đồng thuận kép (Hành vi giỏ hàng $\odot$ Tương đồng phong cách thời trang Text-Visual).
- Van an toàn Gating động thích nghi với độ thưa cực cao ($99.97\%$), tự động khép lại để triệt tiêu nhiễu đa phương thức.
- 100% dòng gradient InfoNCE truyền thẳng vào H0 và H1, không qua projection head trung gian.
- Huấn luyện ở **cùng không gian 256D** với Baseline 256D để đối chứng tuyệt đối công bằng (Ablation Study).

In [ ]:
# Cell 7: Huấn luyện STAIR DCD-Gated (256D, kNN 5-1, Batch 2048)
SELECTED_METHOD = 'dcd_gated'
DATA_ROOT = '/kaggle/data'
YAML_PATH = '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Clothing_550_MMRec.yaml'
LOG_DCD_256D = f"/kaggle/working/logs/breakthrough/clothing_{SELECTED_METHOD}_dim256.log"

run_training_v5_plus(
    key='Amazon2014Clothing_550_MMRec',
    yaml_cfg=YAML_PATH,
    data_root=DATA_ROOT,
    log_path=LOG_DCD_256D,
    method=SELECTED_METHOD,
    epochs=500,
    embedding_dim=256,
    batch_size=2048,
    lr_warmup_epochs=15,
    min_lr=1e-6,
    patience=30,
    tau=0.20,
    alpha_dir=0.50,
    eps=0.08,
    tau_thresh=0.85,
    lambda_cl=0.010,
    gamma_h=0.15,
    warmup_epochs=50,
    mfiles='textual_modality.pkl,visual_modality.pkl',
    num_neighbors='5-1',
)


## 📊 PHẦN 3: Bảng Tổng Kết Đối Sánh Khoa Học & Bóc Tách Đóng Góp (Ablation Study)
Tổng hợp toàn bộ kết quả trên 4 chỉ số khoa học: **Recall@10, Recall@20, NDCG@10, NDCG@20** và tính toán chính xác mức tăng trưởng:
1. **$\Delta$ vs Base 64D:** Mức tăng trưởng so với Baseline chuẩn công bố của tác giả (mốc tham chiếu).
2. **$\Delta$ vs Base 256D:** Mức tăng trưởng thực sự của cơ chế DCD-Gated so với Baseline ở **cùng số chiều 256D**.

In [ ]:
# Cell 9: Bảng Tổng Hợp Đối Sánh Hợp Nhất (PrettyTable)
import os, glob
from prettytable import PrettyTable

# 1. Mốc tham chiếu Baseline 64D Paper Table 4 công bố (@Ep205)
ep_64 = 205
m_64 = {'Recall@10': 0.0596, 'Recall@20': 0.0896, 'NDCG@10': 0.0321, 'NDCG@20': 0.0398}

# 2. Trích xuất Baseline 256D vừa chạy
log_256 = '/kaggle/working/logs/baseline/clothing_stair_baseline_dim256.log'
ep_256, m_256 = extract_best_test(log_256)

# 3. Trích xuất DCD-Gated 256D vừa chạy
log_dcd = '/kaggle/working/logs/breakthrough/clothing_dcd_gated_dim256.log'
ep_dcd, m_dcd = extract_best_test(log_dcd)

table = PrettyTable()
table.field_names = ['Tập dữ liệu', 'Phương pháp', 'Số chiều (Dim)', 'Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20', 'Δ vs Base 64D', 'Δ vs Base 256D']

# Dòng 1: Baseline 64D Paper Reference
base_64_n20 = m_64['NDCG@20']
table.add_row([
    'CLOTHING', f"STAIR Baseline (Paper Table 4)", '64D',
    f"{m_64['Recall@10']:.4f}", f"{m_64['Recall@20']:.4f}",
    f"{m_64['NDCG@10']:.4f}", f"{m_64['NDCG@20']:.4f}",
    '-', '-'
])

# Dòng 2: Baseline 256D
base_256_n20 = None
if m_256 and len(m_256) >= 4:
    base_256_n20 = m_256['NDCG@20']
    gain_256_vs_64 = (base_256_n20 - base_64_n20) / base_64_n20 * 100
    sign_256 = '+' if gain_256_vs_64 >= 0 else ''
    table.add_row([
        'CLOTHING', f"STAIR Baseline [@Ep{ep_256}]", '256D',
        f"{m_256['Recall@10']:.4f}", f"{m_256['Recall@20']:.4f}",
        f"{m_256['NDCG@10']:.4f}", f"{m_256['NDCG@20']:.4f}",
        f"{sign_256}{gain_256_vs_64:.2f}%", '-'
    ])
else:
    table.add_row([
        'CLOTHING', 'STAIR Baseline (Đang chạy...)', '256D',
        '-', '-', '-', '-', '-', '-'
    ])

# Dòng 3: DCD-Gated 256D
if m_dcd and len(m_dcd) >= 4:
    gain_dcd_vs_64 = (m_dcd['NDCG@20'] - base_64_n20) / base_64_n20 * 100
    sign_dcd_64 = '+' if gain_dcd_vs_64 >= 0 else ''
    
    delta_vs_256_str = '-'
    if base_256_n20 is not None:
        gain_dcd_vs_256 = (m_dcd['NDCG@20'] - base_256_n20) / base_256_n20 * 100
        sign_dcd_256 = '+' if gain_dcd_vs_256 >= 0 else ''
        delta_vs_256_str = f"{sign_dcd_256}{gain_dcd_vs_256:.2f}%"
        
    table.add_row([
        'CLOTHING', f"★ STAIR DCD-Gated [@Ep{ep_dcd}]", '256D',
        f"{m_dcd['Recall@10']:.4f}", f"{m_dcd['Recall@20']:.4f}",
        f"{m_dcd['NDCG@10']:.4f}", f"{m_dcd['NDCG@20']:.4f}",
        f"{sign_dcd_64}{gain_dcd_vs_64:.2f}%", delta_vs_256_str
    ])
else:
    table.add_row([
        'CLOTHING', '★ STAIR DCD-Gated (Đang chạy...)', '256D',
        '-', '-', '-', '-', '-', '-'
    ])

print(table)

print('\n' + '=' * 85)
print("🎉 KẾT LUẬN BÓC TÁCH ĐÓNG GÓP (ABLATION STUDY) TRÊN AMAZON CLOTHING:")
print(f"  • Baseline 64D  : NDCG@20 = {base_64_n20:.4f} (Mốc chuẩn Paper Table 4)")
if base_256_n20 is not None:
    print(f"  • Baseline 256D : NDCG@20 = {base_256_n20:.4f} ({sign_256}{gain_256_vs_64:.2f}% vs Base 64D)")
if m_dcd and len(m_dcd) >= 4:
    if base_256_n20 is not None:
        print(f"  • DCD-Gated 256D: NDCG@20 = {m_dcd['NDCG@20']:.4f} ({delta_vs_256_str} vs Base 256D, {sign_dcd_64}{gain_dcd_vs_64:.2f}% vs Base 64D)")
    else:
        print(f"  • DCD-Gated 256D: NDCG@20 = {m_dcd['NDCG@20']:.4f} ({sign_dcd_64}{gain_dcd_vs_64:.2f}% vs Base 64D)")
print('=' * 85)
